# QuakeMind — 10 model ortak protokol (Kaggle)

02 defterinin (Colab) Kaggle sürümü: **aynı bölme, aynı veri, aynı tarif, aynı ölçütler**. Farkları: sonuçlar Drive
yerine `/kaggle/working/QuakeMind_egitim/` (oturum sonunda sürümün çıktısı olarak saklanır), indirilen veri `/tmp`'de,
albumentations 2.0.8 sabit, oturum içi kuyruk (`KUYRUK`) ve 12 saatlik oturum için süre koruması.

**Kullanım** — Settings: Accelerator = **GPU T4 x2** (yalnız ilk GPU kullanılır; Colab'daki T4 ile aynı GPU), Internet =
**açık** (ikisi de telefon doğrulaması ister). **Save Version → Save & Run All** ile arka planda koşar; tarayıcı
kapatılabilir. İlk koşu 0. hücredeki ayarlarla, sonra `KUYRUK`'taki deneyler sırayla. Her koşu
`<deney>/<model>/tohum_<n>/` klasörüne `sonuc.json`, `gecmis.json`, `ornek_ciktilar.jpg`, `model_en_iyi.pth` yazar;
özet `tum_kosular.csv`. Yerel kopya: `araclar/kosu_kaydet.py <sonuc.json>`.

**Girdi:** özel Kaggle veri seti `furkanemrence/quakemind-veri-paketi` (bölme listesi, QQB ve HOTOSM paketleri, HOTOSM
maskeleri; ~2,5 MB). Defter kaynağı 1 MB ile sınırlı olduğu için gömülmedi; 2. hücre dosyaları SHA-1 ile denetler.

Üreteç: `QuakeMind_veri/araclar/defter_dokuz_model.py --kaggle [--ayar "..."]` — hücreleri elle değiştirmeyin,
üreteçten yeniden üretin.

In [ ]:
# 0) Ayarlar — her koşudan önce yalnız bu hücreyi değiştirin
MODEL_ADI = "segformer_mitb4"
#   MiT-B4 kodlayıcılı : segformer_mitb4 · unet_mitb4 · fpn_mitb4 · deeplabv3plus_mitb4 · upernet_mitb4
#   aynı kapasite      : deeplabv3plus_resnet101 · upernet_convnext_small
#   ölçek              : segformer_mitb0 · segformer_mitb2
#   literatür tabanı   : unet_resnet18 (Türkiye çalışmalarında yaygın hafif taban; Hacıefendioğlu vd. 2024)
#   isteğe bağlı       : unet_resnet152 (KATE-PD yayınındaki taban mimari; KOSU_B = True ile yayın kıyası)
TOHUMLAR = [42]              # ilk tur tek tohum; en iyi 3 model için sonra [42, 7, 2026]
PROTOKOL = "P2"              # "P2" önerilen (MEDBFS yalnız dış sınama, SABIR 20) · "P1" = 19 Eylül Koşu A
QQB_EGITIME = False          # True → QQB'nin 339 karosu eğitime eklenir (A3 deneyi; önce yalnız SegFormer ile)
QQB_DIS_SINAMA = True        # 74 QQB karosunda deprem sonrası sağlam binalarda yanlış alarm ölçülür
HOTOSM_EGITIME = None        # None · "dik" (bakış ≤ 30°) · "erken" (bütün açılar) → HOTOSM yıkık bina karoları eğitime
                             #   (0,5 m; pakette yalnız 13 Şubat'a kadarki sahneler var)
HOTOSM_DIS_SINAMA = True     # Adıyaman merkez (hiç eğitilmeyen ilçe): yıkık binaların kaçı bulunuyor
BOLGE_EGITIMI = None         # Tablo 15: "Hatay", "Kahramanmaraş", "Hatay+Kahramanmaraş" ya da il-dışarıda-bırakma
                             #   "haric:Hatay" / "haric:Kahramanmaraş" (hedef il eğitimde yok); None = bütün iller
KAYIP_FOCAL = 0.5            # kayıp = KAYIP_FOCAL × focal + (1 − KAYIP_FOCAL) × dice  (Tablo 21 taraması)
FOCAL_ALFA = 0.25            # focal kaybında hasar piksellerinin ağırlığı; 0,25 hasarı arka plandan az tartar (A-19)
ORNEKLEYICI = None           # None · 0.5 → eğitim partilerinde hasarlı karo payı ~%50 (yerine koyarak örnekleme)
GIRDI = "sonrasi"            # "sonrasi" (10 model tablosu) · "cd_oncesi_sonrasi" = değişim tespiti: KATE-CD çiftleri,
                             #   öncesi + sonrası 6 kanal · "cd_sonrasi" = aynı çiftlerin yalnız sonrası (kıyas tabanı)
KOSU_B = False               # KATE-PD resmî bölmesi (675/78/79) ile ayrıca eğit: yayımlanmış U-Net ResNet-152 kıyası
AGIRLIK_SAKLA = True         # False → model ağırlığı Drive'a yazılmaz (yer kazanır; sonuçlar yine kaydedilir)
GORSEL = True                # ilk tohum için örnek çıktı görseli
DONEM = 50
PARTI = 8
BIRIKIM = 1                  # GPU belleği yetmezse PARTI = 4, BIRIKIM = 2 (etkin parti yine 8; bunu raporda belirtin)
OGRENME_ORANI = 2e-4
ESIK = 0.5
KUYRUK = []                  # Kaggle: ilk koşudan (yukarıdaki ayarlar) sonra sırayla koşulacak ek deneyler; her öğe bu
                             #   hücrenin ayarlarının üstüne yazılır, ör. [dict(HOTOSM_EGITIME="dik"), dict(MODEL_ADI="unet_mitb4")]
KOSU_PAYI_SA = 2.5           # kuyrukta bir deneye ayrılan en uzun süre; 12 saatlik oturuma sığmayacaksa sonraki oturuma kalır
# Bu oturumun ayarları: il dışarıda bırakma deneyi, iki ek tohum (7, 2026).
# Kuyruk gömülü eski ayarları çalıştırdığı için (EKSIKLER J-129) kuyruk öğesinde bütün ayarlar açık yazılı.
MODEL_ADI = "unet_mitb4"
TOHUMLAR = [7, 2026]
BOLGE_EGITIMI = "haric:Hatay"
AGIRLIK_SAKLA = False
GORSEL = False
KUYRUK = [dict(MODEL_ADI="unet_mitb4", TOHUMLAR=[7, 2026], BOLGE_EGITIMI="haric:Kahramanmaraş",
               AGIRLIK_SAKLA=False, GORSEL=False)]


In [ ]:
# 0b) Türetilen ayarlar ve Drive
import os
SABIR = {"P1": 8, "P2": 20}[PROTOKOL]
if GIRDI != "sonrasi":  # değişim tespiti deneyi: yalnız öncesi görüntüsü olan karolar (QQB/HOTOSM'da öncesi yok)
    assert PROTOKOL == "P2" and not BOLGE_EGITIMI, "değişim tespiti deneyi P2 ve bütün illerle koşulur"
    QQB_EGITIME, HOTOSM_EGITIME, QQB_DIS_SINAMA, HOTOSM_DIS_SINAMA = False, None, False, False
KANAL = 6 if GIRDI == "cd_oncesi_sonrasi" else 3
_BOLGE_KISA = {None: "", "Hatay": "_bolge_hatay", "Kahramanmaraş": "_bolge_maras",
               "Hatay+Kahramanmaraş": "_bolge_hatay_maras", "haric:Hatay": "_haric_hatay",
               "haric:Kahramanmaraş": "_haric_maras"}
# Deney adı ayarlardan türetilir: farklı ayar farklı klasöre yazar, hiçbir sonuç üzerine yazılmaz.
DENEY = (PROTOKOL + ("_qqb" if QQB_EGITIME else "") + {None: "", "dik": "_hotdik", "erken": "_hoterken"}[HOTOSM_EGITIME]
         + _BOLGE_KISA[BOLGE_EGITIMI] + ("" if KAYIP_FOCAL == 0.5 else f"_focal{KAYIP_FOCAL:g}")
         + ("" if FOCAL_ALFA == 0.25 else f"_alfa{FOCAL_ALFA:g}")
         + ("" if not ORNEKLEYICI else f"_ornek{round(100 * ORNEKLEYICI)}")
         + {"sonrasi": "", "cd_sonrasi": "_cd_sonrasi", "cd_oncesi_sonrasi": "_cd_oncesi_sonrasi"}[GIRDI])
DUMAN_TESTI = os.environ.get("QM_DUMAN", "0") == "1"
GECICI = os.environ.get("QM_GECICI", "/tmp")   # indirilen veri ve ara dosyalar (sürüm çıktısına girmez)
VERI = os.environ.get("QM_VERI", f"{GECICI}/veri")
if DUMAN_TESTI:
    KOK_CIKTI = os.environ.get("QM_CIKTI", f"{GECICI}/ciktilar")
else:
    # Kaggle: /kaggle/working oturum sonunda sürümün çıktısı olarak saklanır (en çok 20 GB)
    KOK_CIKTI = "/kaggle/working/QuakeMind_egitim"
import time
OTURUM_BASI = globals().get("OTURUM_BASI", time.time())   # 12 saatlik oturum sınırı için (kuyruk hücresi)
MODEL_KLASORU = f"{KOK_CIKTI}/{DENEY}/{MODEL_ADI}"
os.makedirs(VERI, exist_ok=True); os.makedirs(MODEL_KLASORU, exist_ok=True)
print("deney:", DENEY, "| model:", MODEL_ADI, "| tohumlar:", TOHUMLAR, "| sabır:", SABIR, "| çıktı:", MODEL_KLASORU)

In [ ]:
# 1) Kurulum ve GPU denetimi
import subprocess, sys
if not DUMAN_TESTI:
    paketler = ["segmentation-models-pytorch==0.5.0", "albumentations==2.0.8"]   # yerel duman testiyle aynı sürüm
    if QQB_EGITIME or QQB_DIS_SINAMA or HOTOSM_EGITIME or HOTOSM_DIS_SINAMA:
        paketler += ["rasterio", "shapely"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paketler], check=True)
import torch, segmentation_models_pytorch as smp
print("torch", torch.__version__, "| smp", smp.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not DUMAN_TESTI:
    assert torch.cuda.is_available(), "GPU yok: Settings → Accelerator = GPU T4 x2 (telefon doğrulaması gerekir)."

In [ ]:
# 2) Gömülü listeler: ortak bölme (2 194 satır), QQB paketi (karolar + bina poligonları, EPSG:32637),
#    HOTOSM paketi (karolar + Adıyaman dış sınamasındaki yıkık binalar) ve HOTOSM maskeleri (0 / 1 / 255)
import base64, gzip, io, json, zipfile
import pandas as pd
# Kaggle: defter kaynağı 1 MB ile sınırlı → paketler özel Kaggle veri setinde (furkanemrence/quakemind-veri-paketi);
#   içerik yereldekiyle aynı mı diye SHA-1 denetlenir (değerleri üreteç yerel dosyalardan hesapladı).
import glob, hashlib, os
PAKET = os.environ.get("QM_PAKET") or os.path.dirname(
    (glob.glob("/kaggle/input/**/hotosm_paket.json", recursive=True) or [""])[0])
assert PAKET, "veri paketi bulunamadı: furkanemrence/quakemind-veri-paketi defterin Input bölümüne eklenmeli"
PAKET_SHA1 = {'bolme_listesi.csv': '83f188764a3a6c403a5f986b351503a4e1b81f18', 'qqb_paket.json': '54605f4cc4c38e4c473a320249e2f6a87a2bec1b', 'hotosm_paket.json': 'ed5751bc8cd2b6294176180e1f0c57d48e2ceca3', 'hotosm_maskeler': '992f99370f6e92088be6a5a47c3968a0f4c70d8c'}
def _oku(ad):
    b = open(f"{PAKET}/{ad}", "rb").read()
    assert hashlib.sha1(b).hexdigest() == PAKET_SHA1[ad], ("veri paketi yereldekinden farklı:", ad)
    return b
LISTE_B64 = base64.b64encode(gzip.compress(_oku("bolme_listesi.csv"))).decode()
QQB_B64 = base64.b64encode(gzip.compress(_oku("qqb_paket.json"))).decode()
HOT_B64 = base64.b64encode(gzip.compress(_oku("hotosm_paket.json"))).decode()
# Maskeler: Kaggle zip'i veri setinde açar (hotosm_maskeler/*.png), yerelde zip; özet ikisinde de içerikten
if os.path.isdir(f"{PAKET}/hotosm_maskeler"):
    _mk = {os.path.basename(p): open(p, "rb").read() for p in glob.glob(f"{PAKET}/hotosm_maskeler/*.png")}
else:
    with zipfile.ZipFile(f"{PAKET}/hotosm_maskeler.zip") as _z:
        _mk = {n: _z.read(n) for n in _z.namelist() if not n.endswith("/")}
_h = hashlib.sha1()
for _n in sorted(_mk):
    _h.update(_n.encode()); _h.update(_mk[_n])
assert _h.hexdigest() == PAKET_SHA1["hotosm_maskeler"], "HOTOSM maskeleri yereldekinden farklı"
_bz = io.BytesIO()
with zipfile.ZipFile(_bz, "w") as _z:
    for _n in sorted(_mk):
        _z.writestr(_n, _mk[_n])
HOT_MASKE_B64 = base64.b64encode(_bz.getvalue()).decode()
print("veri paketi:", PAKET, "| SHA-1 denetimi geçti |", len(_mk), "HOTOSM maskesi")
liste = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(LISTE_B64))), dtype=str, keep_default_na=False)
liste["olcut"] = liste["olcut"].astype(int)
qqb = json.loads(gzip.decompress(base64.b64decode(QQB_B64)))
hot = (json.loads(gzip.decompress(base64.b64decode(HOT_B64))) if HOT_B64
       else {"karolar": [], "yikik_dis_sinama": [], "kenar_m": 256.0, "gsd": 0.5})
HOT_MASKE = zipfile.ZipFile(io.BytesIO(base64.b64decode(HOT_MASKE_B64))) if HOT_MASKE_B64 else None
if not hot["karolar"]:
    HOTOSM_EGITIME, HOTOSM_DIS_SINAMA = None, False
print(len(liste), "satır | QQB:", pd.Series([k["grup"] for k in qqb["karolar"]]).value_counts().to_dict(),
      "| yıkık bina:", len(qqb["yikik"]), "| dış sınamadaki sağlam bina:", len(qqb["saglam_dis_sinama"]),
      "| belirsiz (255):", len(qqb.get("belirsiz", [])))
print("HOTOSM:", pd.Series([f'{k["grup"]}/{"dik" if str(k["dik"]) == "1" else "eğik"}' for k in hot["karolar"]]).value_counts().to_dict(),
      "| Adıyaman'daki yıkık bina:", len(hot["yikik_dis_sinama"]))

In [ ]:
# 3) Veri indirme: KATE-PD, KATE-CD (Hugging Face) · MEDBFS (Zenodo; yalnız görüntü ve etiket) · QQB için Maxar (S3)
import os, shutil, subprocess, time
def indir(url, hedef, deneme=5):
    for k in range(deneme):
        r = subprocess.run(["wget", "-q", "-c", "-O", hedef, url] if shutil.which("wget")
                           else ["curl", "-sSfL", "-C", "-", "-o", hedef, url])
        if r.returncode == 0 and os.path.getsize(hedef) > 0:
            return
        time.sleep(10 * (k + 1))
    raise RuntimeError("indirilemedi: " + url)
HF = "https://huggingface.co/datasets/CSCRS/{}/resolve/main/data/{}-00000-of-00001.parquet"
for ds in ("kate-pd", "kate-cd"):
    os.makedirs(f"{VERI}/{ds}", exist_ok=True)
    for sp in ("train", "validation", "test"):
        h = f"{VERI}/{ds}/{sp}.parquet"
        if not os.path.exists(h):
            indir(HF.format(ds, sp), h)
MEDBFS_KOK = f"{VERI}/medbfs"
if not os.path.exists(f"{MEDBFS_KOK}/Turkey/Label") and not os.path.exists(f"{MEDBFS_KOK}/Label"):
    os.makedirs(MEDBFS_KOK, exist_ok=True)
    t = time.time()
    indir("https://zenodo.org/records/11647956/files/Turkey.zip?download=1", f"{GECICI}/Turkey.zip")
    subprocess.run(["unzip", "-q", "-o", f"{GECICI}/Turkey.zip", "Turkey/Image/*", "Turkey/Label/*", "-d", MEDBFS_KOK], check=True)
    os.remove(f"{GECICI}/Turkey.zip")
    print("MEDBFS indirildi:", round(time.time() - t), "sn")
if QQB_EGITIME or QQB_DIS_SINAMA:
    os.makedirs(f"{VERI}/qqb/maxar", exist_ok=True)
    for q in sorted({k["tif"] for k in qqb["karolar"]}):
        h = f"{VERI}/qqb/maxar/{q}_visual.tif"
        if not os.path.exists(h):
            indir(qqb["maxar_url"].format(q=q), h)   # Maxar Open Data, katalog 1040010082698700, CC BY-NC 4.0
print("veri hazır")

In [ ]:
# 4) Bu deneyde her karonun rolü (egitim / dogrulama / sinama / dis_medbfs / dis_qqb) ve belleğe alma
import numpy as np, pyarrow.parquet as pq, gc
from PIL import Image
from skimage.exposure import match_histograms

IKIZLI = set(liste.ikiz[liste.kume == "KATE-CD"])   # öncesi görüntüsü olan (KATE-CD'de ikizi bulunan) KATE-PD karoları
def rol_ata(r):
    if r.bolme == "tampon":
        return ""
    if GIRDI != "sonrasi":  # değişim tespiti: yalnız öncesi görüntüsü olan sonrası karoları; öncesi girdinin parçası
        if r.goruntu != "sonrasi" or not (r.kume == "MEDBFS" or r.ornek in IKIZLI):
            return ""
    if PROTOKOL == "P2" and r.kume == "MEDBFS":
        return "dis_medbfs" if (r.bolme in ("val", "test") and r.olcut == 1) else ""
    if r.bolme == "train":
        return "egitim"
    if r.olcut != 1:
        return ""
    return "dogrulama" if r.bolme == "val" else "sinama"
liste["rol"] = [rol_ata(r) for r in liste.itertuples()]
qk = pd.DataFrame(qqb["karolar"])
qq = pd.DataFrame(dict(ornek="qqb/" + qk.ornek, kume="QQB", goruntu="sonrasi", bolme=qk.grup, olcut=1, grup="",
                       il="Gaziantep", ilce="İslahiye", resmi_bolme="", kaynak="qqb_maxar", hasar_orani="", lat="", lon="",
                       kaynak_dosya=qk.ornek, etiket_dosya="", ikiz=""))
qq["rol"] = [("egitim" if QQB_EGITIME else "") if g == "train" else ("dis_qqb" if QQB_DIS_SINAMA else "") for g in qk.grup]
hk = pd.DataFrame(hot["karolar"], columns=["ornek", "grup", "dik", "il", "ilce"])
hq = pd.DataFrame(dict(ornek="hotosm/" + hk.ornek, kume="HOTOSM", goruntu="sonrasi", bolme=hk.grup, olcut=1, grup="",
                       il=hk.il, ilce=hk.ilce, resmi_bolme="", kaynak="hotosm_maxar", hasar_orani="", lat="", lon="",
                       kaynak_dosya=hk.ornek, etiket_dosya="", ikiz=""))
def hot_rol(g, dik):
    if g == "train":
        return "egitim" if HOTOSM_EGITIME == "erken" or (HOTOSM_EGITIME == "dik" and str(dik) == "1") else ""
    return "dis_hotosm" if (g == "dis_sinama" and HOTOSM_DIS_SINAMA) else ""
hq["rol"] = [hot_rol(g, d) for g, d in zip(hk.grup, hk.dik)]
tum = pd.concat([liste, qq, hq], ignore_index=True)
if BOLGE_EGITIMI:  # Tablo 15: eğitim ve model seçimi yalnız seçilen illerde; sınama bütün illerde, il il raporlanır
    if BOLGE_EGITIMI.startswith("haric:"):  # il-dışarıda-bırakma: hedef il eğitimde ve model seçiminde yok (A-20)
        kisit = tum.rol.isin(["egitim", "dogrulama"]) & tum.il.eq(BOLGE_EGITIMI.split(":", 1)[1])
    else:
        kisit = tum.rol.isin(["egitim", "dogrulama"]) & ~tum.il.isin(BOLGE_EGITIMI.split("+"))
    tum.loc[kisit, "rol"] = ""
kullan = tum[tum.rol != ""].reset_index(drop=True)
if DUMAN_TESTI:  # her gruptan birkaç örnek
    kullan = kullan.groupby(["rol", "kume", "goruntu", "resmi_bolme"]).head(3).reset_index(drop=True)
print(kullan.groupby(["rol", "kume", "goruntu"]).size().unstack(fill_value=0))

def pil(b):
    return Image.open(io.BytesIO(b["bytes"] if isinstance(b, dict) else b))
def maske(im):
    a = np.asarray(im)
    if a.ndim == 3:
        a = a.max(axis=2)
    return (a > 0).astype(np.uint8)
def medbfs_yol(ad):
    for p in (f"{MEDBFS_KOK}/{ad}", f"{MEDBFS_KOK}/{ad.replace('Turkey/', '', 1)}"):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(ad)

if kullan.kume.isin(["QQB", "HOTOSM"]).any():
    import rasterio
    from rasterio.enums import Resampling
    from rasterio.features import rasterize
    from rasterio.transform import from_bounds as donusum_kutudan
    from rasterio.windows import from_bounds as pencere_kutudan
    from shapely.geometry import box, shape
    YIKIK = [shape(g) for g in qqb["yikik"]]
    SAGLAM = [shape(g) for g in qqb["saglam_dis_sinama"]]
    BELIRSIZ = [shape(g) for g in qqb.get("belirsiz", [])]   # A-18: maskede 255
    KARO_Q = {k["ornek"]: k for k in qqb["karolar"]}
    def _rasterle(parcalar, tr, tip):
        parcalar = [(g, v) for g, v in parcalar if not g.is_empty]
        return (rasterize(parcalar, out_shape=(512, 512), transform=tr, fill=0, dtype=tip) if parcalar
                else np.zeros((512, 512), tip))
    def qqb_karo(ad, kimlikli=False):
        """Maxar 0,31 m -> 0,375 m, 512 x 512 (qqb_karola.py ile aynı); maske: yıkık = 1, belirsiz = 255, gerisi 0."""
        k = KARO_Q[ad]; x0, y0, K = float(k["x0"]), float(k["y0"]), qqb["kenar_m"]
        with rasterio.open(f"{VERI}/qqb/maxar/{k['tif']}_visual.tif") as src:
            a = src.read([1, 2, 3], window=pencere_kutudan(x0, y0, x0 + K, y0 + K, src.transform),
                         out_shape=(3, 512, 512), resampling=Resampling.average)
        kutu = box(x0, y0, x0 + K, y0 + K); tr = donusum_kutudan(x0, y0, x0 + K, y0 + K, 512, 512)
        yk = [(g.intersection(kutu), j + 1) for j, g in enumerate(YIKIK) if g.intersects(kutu)]
        bl = [(g.intersection(kutu), 255) for g in BELIRSIZ if g.intersects(kutu)]
        msk = _rasterle(bl + [(g, 1) for g, _ in yk], tr, "uint8")   # önce 255, sonra 1 (çakışmada yıkık kazanır)
        kimlik = None
        if kimlikli:  # bina düzeyi ölçüm için bina kimlik rasterleri (sağlam, yıkık)
            sg = [(g.intersection(kutu), j + 1) for j, g in enumerate(SAGLAM) if g.intersects(kutu)]
            kimlik = (_rasterle(sg, tr, "uint16"), _rasterle(yk, tr, "uint16"))
        return np.transpose(a, (1, 2, 0)), msk, kimlik
    GDAL_ORTAM = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif", VSI_CACHE="TRUE",
                      GDAL_HTTP_MULTIRANGE="YES", GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES", GDAL_HTTP_MAX_RETRY="5",
                      GDAL_HTTP_RETRY_DELAY="3")
    KARO_H = {k["ornek"]: k for k in hot["karolar"]}
    YIKIK_H = [shape(g) for g in hot["yikik_dis_sinama"]]
    def hot_goruntu(ad):
        """Maxar deprem sonrası, 0,5 m, 512 x 512 (hotosm_karola.py ile aynı pencere); oturum içinde önbelleğe yazılır."""
        onbellek = f"{VERI}/hotosm/{ad}.png"
        if os.path.exists(onbellek):
            return np.asarray(Image.open(onbellek).convert("RGB"))
        k = KARO_H[ad]; x0, y0, K = float(k["x0"]), float(k["y0"]), hot["kenar_m"]
        with rasterio.Env(**GDAL_ORTAM), rasterio.open("/vsicurl/" + k["url"]) as src:
            yontem = Resampling.average if src.res[0] < hot["gsd"] else Resampling.bilinear
            a = src.read([1, 2, 3], window=pencere_kutudan(x0, y0, x0 + K, y0 + K, src.transform),
                         out_shape=(3, 512, 512), resampling=yontem)
        img = np.transpose(a, (1, 2, 0))
        os.makedirs(f"{VERI}/hotosm", exist_ok=True)
        Image.fromarray(img).save(onbellek)
        return img
    def hot_karo(ad, kimlikli=False):
        """Görüntü COG'dan, maske pakette (yıkık = 1, diğer OSM binaları = 255, gerisi 0)."""
        img = hot_goruntu(ad)
        msk = np.asarray(Image.open(io.BytesIO(HOT_MASKE.read(f"{ad}.png"))))
        kimlik = None
        if kimlikli:  # yalnız yıkık bina kimlikleri (HOTOSM'da doğrulanmış sağlam bina listesi yok)
            k = KARO_H[ad]; x0, y0, K = float(k["x0"]), float(k["y0"]), hot["kenar_m"]
            kutu = box(x0, y0, x0 + K, y0 + K); tr = donusum_kutudan(x0, y0, x0 + K, y0 + K, 512, 512)
            yk = [(g.intersection(kutu), j + 1) for j, g in enumerate(YIKIK_H) if g.intersects(kutu)]
            kimlik = (np.zeros((512, 512), np.uint16), _rasterle(yk, tr, "uint16"))
        return img, msk, kimlik
    gerekli = [r.kaynak_dosya for r in kullan.itertuples()
               if r.kume == "HOTOSM" and not os.path.exists(f"{VERI}/hotosm/{r.kaynak_dosya}.png")]
    if gerekli:  # HOTOSM pencereleri paralel okunur (her karo ~0,2–0,5 MB)
        from concurrent.futures import ThreadPoolExecutor
        t_ = time.time()
        with ThreadPoolExecutor(8) as ex:
            list(ex.map(hot_goruntu, gerekli))
        print("HOTOSM görüntüleri okundu:", len(gerekli), "karo,", round(time.time() - t_), "sn")

kpd = {}
for sp in ("train", "validation", "test"):
    for r in pq.read_table(f"{VERI}/kate-pd/{sp}.parquet", columns=["name", "image", "mask"]).to_pylist():
        kpd[r["name"]] = r
kcd = {}
for sp in ("train", "validation", "test"):
    for i, r in enumerate(pq.read_table(f"{VERI}/kate-cd/{sp}.parquet", columns=["pre_image"]).to_pylist()):
        kcd[f"{sp}/{i}"] = r["pre_image"]

N = len(kullan)
X = np.zeros((N, 512, 512, KANAL), np.uint8)   # 6 kanal: sonrası RGB + öncesi RGB (değişim tespiti)
ONCESI_CD = {r.ikiz: r.kaynak_dosya for r in liste.itertuples() if r.kume == "KATE-CD"}
ONCESI_MB = {r.ikiz: r.kaynak_dosya for r in liste.itertuples() if r.kume == "MEDBFS" and r.goruntu == "oncesi"}
Y = np.zeros((N, 512, 512), np.uint8)
BINA = {}
sonrasi_rgb = {}
t = time.time()
sira = list(kullan.index[kullan.goruntu == "sonrasi"]) + list(kullan.index[kullan.goruntu == "oncesi"])
for i in sira:
    r = kullan.loc[i]
    if r.kume == "QQB":
        img, msk, kimlik = qqb_karo(r.kaynak_dosya, kimlikli=r.rol == "dis_qqb")
        if kimlik is not None:
            BINA[i] = kimlik
    elif r.kume == "HOTOSM":
        img, msk, kimlik = hot_karo(r.kaynak_dosya, kimlikli=r.rol == "dis_hotosm")
        if kimlik is not None:
            BINA[i] = kimlik
    elif r.kume == "KATE-PD":
        k = kpd[r.kaynak_dosya]
        img = np.asarray(pil(k["image"]).convert("RGB")); msk = maske(pil(k["mask"]))
    elif r.kume == "MEDBFS" and r.goruntu == "sonrasi":
        img = np.asarray(Image.open(medbfs_yol(r.kaynak_dosya)).convert("RGB"))
        msk = maske(Image.open(medbfs_yol(r.etiket_dosya)))
    else:  # öncesi görüntüsü: boş maske, renkleri ikiz sonrası görüntüye uydurulur
        if r.kume == "KATE-CD":
            img = np.asarray(pil(kcd[r.kaynak_dosya]).convert("RGB"))
        else:
            img = np.asarray(Image.open(medbfs_yol(r.kaynak_dosya)).convert("RGB"))
        ikiz = sonrasi_rgb.get(r.ikiz)
        if ikiz is None:  # ikiz bu deneyde yüklenmediyse doğrudan oku
            if r.ikiz.startswith("katepd/"):
                ikiz = np.asarray(pil(kpd[r.ikiz.split("/", 1)[1]]["image"]).convert("RGB"))
            else:
                ikiz = np.asarray(Image.open(medbfs_yol("Turkey/Image/Post-disaster/" + r.ikiz.split("/", 1)[1])).convert("RGB"))
        img = np.clip(match_histograms(img, ikiz, channel_axis=-1), 0, 255).astype(np.uint8)
        msk = np.zeros((512, 512), np.uint8)
    assert img.shape == (512, 512, 3) and msk.shape == (512, 512), (r.ornek, img.shape, msk.shape)
    X[i, ..., :3] = img; Y[i] = msk
    if KANAL == 6:  # öncesi görüntüsü; renkleri sonrasına uydurulur (öncesi negatiflerindeki gibi)
        on = (np.asarray(pil(kcd[ONCESI_CD[r.ornek]]).convert("RGB")) if r.kume == "KATE-PD"
              else np.asarray(Image.open(medbfs_yol(ONCESI_MB[r.ornek])).convert("RGB")))
        X[i, ..., 3:] = np.clip(match_histograms(on, img, channel_axis=-1), 0, 255).astype(np.uint8)
    if r.goruntu == "sonrasi":
        sonrasi_rgb[r.ornek] = img
del sonrasi_rgb, kpd, kcd
gc.collect()
print(N, "karo belleğe alındı,", round(time.time() - t), "sn | bellek ~", round((X.nbytes + Y.nbytes) / 1e9, 2), "GB")

In [ ]:
# 5) Modeller, veri artırımı, ölçütler ve eğitim döngüsü
import random, copy
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from scipy import ndimage as ndi
from torch.utils.data import Dataset, DataLoader

MIMARILER = {
    "segformer_mitb4": lambda w: smp.Segformer(encoder_name="mit_b4", encoder_weights=w, classes=1, in_channels=KANAL),
    "unet_mitb4": lambda w: smp.Unet(encoder_name="mit_b4", encoder_weights=w, classes=1, in_channels=KANAL),
    "fpn_mitb4": lambda w: smp.FPN(encoder_name="mit_b4", encoder_weights=w, classes=1, in_channels=KANAL),
    "deeplabv3plus_mitb4": lambda w: smp.DeepLabV3Plus(encoder_name="mit_b4", encoder_weights=w, classes=1, in_channels=KANAL),
    "upernet_mitb4": lambda w: smp.UPerNet(encoder_name="mit_b4", encoder_weights=w, classes=1, in_channels=KANAL),
    "deeplabv3plus_resnet101": lambda w: smp.DeepLabV3Plus(encoder_name="resnet101", encoder_weights=w, classes=1, in_channels=KANAL),
    "upernet_convnext_small": lambda w: smp.UPerNet(encoder_name="tu-convnext_small", encoder_weights=w, classes=1, in_channels=KANAL),
    "segformer_mitb0": lambda w: smp.Segformer(encoder_name="mit_b0", encoder_weights=w, classes=1, in_channels=KANAL),
    "segformer_mitb2": lambda w: smp.Segformer(encoder_name="mit_b2", encoder_weights=w, classes=1, in_channels=KANAL),
    "unet_resnet18": lambda w: smp.Unet(encoder_name="resnet18", encoder_weights=w, classes=1, in_channels=KANAL),
    "unet_resnet152": lambda w: smp.Unet(encoder_name="resnet152", encoder_weights=w, classes=1, in_channels=KANAL),
}
assert MODEL_ADI in MIMARILER, "MODEL_ADI şunlardan biri olmalı: " + ", ".join(MIMARILER)

def tohumla(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    # albumentations 2.x global tohumu dikkate almaz (22 Eylül ölçüldü): artırım ayrıca tohumlanır; DataLoader
    # işçileri torch'un işçi tohumundan (_isci_tohumu) — aynı tohum aynı artırımı, her dönem ve işçi farklı artırımı verir
    egitim_donusumu.set_random_seed(s); deger_donusumu.set_random_seed(s)

def _isci_tohumu(_):
    w = torch.utils.data.get_worker_info()
    w.dataset.d.set_random_seed(torch.initial_seed() % 2**32)

ORT, SS = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# değişim tespitinde öncesi görüntüsü aynı geometrik ve renk dönüşümünü alır (additional_targets)
_EK = {"additional_targets": {"oncesi": "image"}} if KANAL == 6 else {}
egitim_donusumu = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.Affine(translate_percent=(-0.05, 0.05), scale=(0.9, 1.1), rotate=(-20, 20), p=0.5, border_mode=4),
    A.RandomResizedCrop(size=(512, 512), scale=(0.7, 1.0), ratio=(0.9, 1.1), p=0.3),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.5),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.15), A.GaussNoise(std_range=(0.02, 0.1), p=0.15),
    A.Normalize(mean=ORT, std=SS), ToTensorV2(),
], **_EK)
deger_donusumu = A.Compose([A.Normalize(mean=ORT, std=SS), ToTensorV2()], **_EK)

class Karolar(Dataset):
    def __init__(self, idx, donusum):
        self.idx, self.d = np.asarray(idx), donusum
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        if KANAL == 6:
            a = self.d(image=np.ascontiguousarray(X[i][..., :3]), oncesi=np.ascontiguousarray(X[i][..., 3:]), mask=Y[i])
            return torch.cat([a["image"], a["oncesi"]], 0), a["mask"].float().unsqueeze(0), i
        a = self.d(image=X[i], mask=Y[i])
        return a["image"], a["mask"].float().unsqueeze(0), i

def yukleyici(idx, egitim):
    ornekleyici = None
    if egitim and ORNEKLEYICI:  # hasarlı karo payı ~ORNEKLEYICI olacak şekilde yerine koyarak örnekleme
        hasarli = np.array([(Y[i] == 1).any() for i in idx])
        w = np.where(hasarli, ORNEKLEYICI / max(int(hasarli.sum()), 1), (1 - ORNEKLEYICI) / max(int((~hasarli).sum()), 1))
        ornekleyici = torch.utils.data.WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),
                                                             num_samples=len(idx), replacement=True)
    return DataLoader(Karolar(idx, egitim_donusumu if egitim else deger_donusumu), batch_size=PARTI,
                      shuffle=egitim and ornekleyici is None, sampler=ornekleyici,
                      num_workers=0 if DUMAN_TESTI else 2, pin_memory=not DUMAN_TESTI,
                      worker_init_fn=None if DUMAN_TESTI else _isci_tohumu,
                      drop_last=egitim and len(idx) > PARTI)

cihaz = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@torch.no_grad()
def tahmin_et(model, idx):
    """Karo başına ikili tahmin maskesi (uint8)."""
    model.eval(); out = {}
    for x, _, ii in yukleyici(idx, False):
        with torch.autocast(device_type=cihaz.type, dtype=torch.float16, enabled=cihaz.type == "cuda"):
            p = torch.sigmoid(model(x.to(cihaz, non_blocking=True))).float()
        for j, i in enumerate(ii.tolist()):
            out[i] = (p[j, 0] > ESIK).cpu().numpy().astype(np.uint8)
    return out

def olcutler(tah, idx):
    """Tüm pikseller üzerinden (global) ölçütler + karo başına IoU ortalaması. 255 (yok say) pikseller sayılmaz.
    Boş kümede None."""
    if len(idx) == 0:
        return None
    tp = fp = fn = tn = 0; karo_iou = []
    for i in idx:
        v = Y[i] != 255
        p, g = tah[i].astype(bool) & v, Y[i] == 1
        a, b, c = int((p & g).sum()), int((p & ~g).sum()), int((~p & g).sum())
        tp += a; fp += b; fn += c; tn += int((~p & ~g & v).sum())
        if g.any() or p.any():
            karo_iou.append(a / max(a + b + c, 1))
    e = 1e-9
    return dict(n_karo=len(idx), iou_hasar=tp / (tp + fp + fn + e), kesinlik=tp / (tp + fp + e),
                duyarlilik=tp / (tp + fn + e), f1=2 * tp / (2 * tp + fp + fn + e), ozgulluk=tn / (tn + fp + e),
                dogruluk=(tp + tn) / (tp + tn + fp + fn + e), iou_arka=tn / (tn + fp + fn + e),
                karo_ortalama_iou=float(np.mean(karo_iou)) if karo_iou else None)

def yanlis_alarm(tah, idx, min_piksel=64):
    """Hasarsız karolarda: hasar denen piksel oranı ve >= min_piksel hasar denen karo oranı."""
    if len(idx) == 0:
        return None
    hasar = {i: tah[i].astype(bool) & (Y[i] != 255) for i in idx}
    px = [float(hasar[i].sum() / max(int((Y[i] != 255).sum()), 1)) for i in idx]
    return dict(n_karo=len(idx), yanlis_piksel_orani=float(np.mean(px)),
                alarm_veren_karo_orani=float(np.mean([hasar[i].sum() >= min_piksel for i in idx])))

def gsd(i):
    k = kullan.kaynak[i]
    return 0.375 if k in ("maxar", "qqb_maxar") else 0.5 if k in ("pleiades", "hotosm_maxar") else 0.751

def bilesenler(tah, idx, min_px=10):
    """Bağlantılı bileşen (≈ bina kümesi) düzeyi: boyuta göre bulunma oranı ve bileşen kesinliği."""
    if len(idx) == 0:
        return None
    boy = {"kucuk (<50 m2)": [0, 0], "orta (50-200 m2)": [0, 0], "buyuk (>200 m2)": [0, 0]}
    kap25 = top = tah_top = yanlis = 0
    for i in idx:
        p, g = tah[i].astype(bool) & (Y[i] != 255), Y[i] == 1
        lg, ng = ndi.label(g, structure=np.ones((3, 3)))
        for k in range(1, ng + 1):
            m = lg == k
            alan = m.sum() * gsd(i) ** 2
            s = "kucuk (<50 m2)" if alan < 50 else "orta (50-200 m2)" if alan <= 200 else "buyuk (>200 m2)"
            boy[s][0] += int((p & m).any()); boy[s][1] += 1
            kap25 += int((p & m).sum() >= 0.25 * m.sum()); top += 1
        lp, npr = ndi.label(p, structure=np.ones((3, 3)))
        for k in range(1, npr + 1):
            m = lp == k
            if m.sum() < min_px:
                continue
            tah_top += 1; yanlis += int(not (m & g).any())
    return dict(boyuta_gore={k: dict(bulunan=v[0], toplam=v[1], oran=(v[0] / v[1] if v[1] else None)) for k, v in boy.items()},
                bina_duyarliligi_ortusme=sum(v[0] for v in boy.values()) / max(top, 1),
                bina_duyarliligi_yuzde25=kap25 / max(top, 1), gercek_bilesen=top, tahmin_bilesen=tah_top,
                yanlis_alarm_bilesen=yanlis, bilesen_kesinligi=(tah_top - yanlis) / max(tah_top, 1))

def qqb_bina(tah, idx, oran=0.25):
    """QQB dış sınama: sağlam binalarda yanlış alarm, yıkık binalarda bulunma (bina ayak izinin >= %25'i hasar)."""
    if len(idx) == 0:
        return None
    sg, yk = {}, {}
    for i in idx:
        p = tah[i].astype(np.float64).ravel()
        for ids, depo in zip(BINA[i], (sg, yk)):
            v = ids.ravel()
            if not v.any():
                continue
            toplam = np.bincount(v); isabet = np.bincount(v, weights=p, minlength=len(toplam))
            for j in np.nonzero(toplam)[0]:
                if j:
                    a = depo.setdefault(int(j), [0.0, 0]); a[0] += isabet[j]; a[1] += int(toplam[j])
    so = [a[0] / a[1] for a in sg.values()]; yo = [a[0] / a[1] for a in yk.values()]
    ort = lambda v: float(np.mean(v)) if v else None
    return dict(saglam_bina=len(so), saglam_yanlis_alarm_bina=int(sum(o >= oran for o in so)),
                saglam_yanlis_alarm_orani=ort([o >= oran for o in so]), saglam_herhangi_piksel_orani=ort([o > 0 for o in so]),
                yikik_bina=len(yo), yikik_bulunan=int(sum(o >= oran for o in yo)), yikik_duyarlilik=ort([o >= oran for o in yo]))

def morfoloji(tah, idx, seviye):
    """Seviye k: k yinelemeli açma, ardından k yinelemeli kapama (3 x 3). Tablo 21'deki 0 / 1 / 2."""
    st = np.ones((3, 3), bool)
    return {i: ndi.binary_closing(ndi.binary_opening(tah[i].astype(bool), st, iterations=seviye), st,
                                  iterations=seviye).astype(np.uint8) for i in idx}

def egit(egitim_idx, dogrulama_idx, tohum, klasor):
    assert len(dogrulama_idx), "doğrulama kümesi boş"
    tohumla(tohum)
    model = MIMARILER[MODEL_ADI](None if DUMAN_TESTI else "imagenet").to(cihaz)
    # ignore_index=255: "yok say" pikselleri kayba girmez (255 olmayan karolarda sonuç öncekiyle aynı)
    dice = smp.losses.DiceLoss(mode="binary", ignore_index=255)
    focal = smp.losses.FocalLoss(mode="binary", alpha=FOCAL_ALFA, gamma=2.0, ignore_index=255)
    kayip_f = lambda p, t: KAYIP_FOCAL * focal(p, t) + (1 - KAYIP_FOCAL) * dice(p, t)
    opt = torch.optim.AdamW(model.parameters(), lr=OGRENME_ORANI, weight_decay=1e-2)
    cz = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=5, T_mult=2, eta_min=1e-6)
    olcek = torch.amp.GradScaler("cuda", enabled=cihaz.type == "cuda")
    en_iyi, en_iyi_donem, bekle, gecmis, en_iyi_durum = -1.0, 0, 0, [], None
    agirlik_yolu = (klasor if AGIRLIK_SAKLA else f"{GECICI}/agirlik_gecici") + "/model_en_iyi.pth"
    os.makedirs(os.path.dirname(agirlik_yolu), exist_ok=True)
    yk = yukleyici(egitim_idx, True)
    donem_sayisi = 1 if DUMAN_TESTI else DONEM
    for d in range(1, donem_sayisi + 1):
        model.train(); t0 = time.time(); top = 0.0; n = 0
        opt.zero_grad(set_to_none=True)
        for k, (x, y, _) in enumerate(yk):
            x, y = x.to(cihaz, non_blocking=True), y.to(cihaz, non_blocking=True)
            with torch.autocast(device_type=cihaz.type, dtype=torch.float16, enabled=cihaz.type == "cuda"):
                kayip = kayip_f(model(x), y) / BIRIKIM
            olcek.scale(kayip).backward()
            if (k + 1) % BIRIKIM == 0 or k + 1 == len(yk):
                olcek.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                olcek.step(opt); olcek.update(); opt.zero_grad(set_to_none=True)
            top += kayip.item() * BIRIKIM; n += 1
            if DUMAN_TESTI and k >= 1:
                break
        cz.step()
        dg = olcutler(tahmin_et(model, dogrulama_idx), dogrulama_idx)
        gecmis.append(dict(donem=d, egitim_kaybi=top / max(n, 1), dogrulama_iou=dg["iou_hasar"], dogrulama_f1=dg["f1"],
                           ogrenme_orani=opt.param_groups[0]["lr"], sure_sn=round(time.time() - t0, 1)))
        iyilesti = dg["iou_hasar"] > en_iyi
        if iyilesti:
            en_iyi, en_iyi_donem, bekle = dg["iou_hasar"], d, 0
            en_iyi_durum = copy.deepcopy(model.state_dict())
            torch.save(en_iyi_durum, agirlik_yolu)
        else:
            bekle += 1
        print(f"[{MODEL_ADI} t{tohum}] dönem {d}/{donem_sayisi} kayıp {top / max(n, 1):.4f} | doğrulama IoU_hasar "
              f"{dg['iou_hasar']:.4f} F1 {dg['f1']:.4f} | {gecmis[-1]['sure_sn']:.0f} sn" + (" *" if iyilesti else ""), flush=True)
        json.dump(gecmis, open(f"{klasor}/gecmis.json", "w", encoding="utf-8"))
        if bekle >= SABIR:
            print(f"erken durdurma; en iyi dönem {en_iyi_donem} (IoU {en_iyi:.4f})"); break
    model.load_state_dict(en_iyi_durum)
    return model, dict(en_iyi_donem=en_iyi_donem, en_iyi_dogrulama_iou=en_iyi, donem_sayisi=len(gecmis), gecmis=gecmis)
print("hazır")

In [ ]:
# 6) Eğitim ve değerlendirme — her tohum için
import datetime, platform
import matplotlib.pyplot as plt
from torch.utils.flop_counter import FlopCounterMode
try:
    from IPython.display import Image as IPImage, display
except ImportError:  # yerel duman testi
    IPImage = display = None

secim = lambda r: kullan.index[kullan.rol == r].tolist()
tr, va, te, dm, dq = secim("egitim"), secim("dogrulama"), secim("sinama"), secim("dis_medbfs"), secim("dis_qqb")
dh = secim("dis_hotosm")
S = (kullan.goruntu == "sonrasi").values
KPD = (kullan.kume == "KATE-PD").values
alt = lambda idx, m: [i for i in idx if m[i]]
print("eğitim", len(tr), "| doğrulama", len(va), "| sınama", len(te), "| dış MEDBFS", len(dm), "| dış QQB", len(dq),
      "| dış HOTOSM (Adıyaman)", len(dh))
print("eğitimdeki kümeler:", kullan.loc[tr].groupby(["kume", "goruntu"]).size().to_dict())

def maliyet_olc(m):
    m = m.eval()
    x = torch.randn(1, KANAL, 512, 512, device=cihaz)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        m(x)
    def zamanla(n=3 if DUMAN_TESTI else 50):
        with torch.no_grad():
            for _ in range(3): m(x)
            if cihaz.type == "cuda": torch.cuda.synchronize()
            t = time.time()
            for _ in range(n): m(x)
            if cihaz.type == "cuda": torch.cuda.synchronize()
        return (time.time() - t) / n * 1000
    return dict(parametre_m=round(sum(p.numel() for p in m.parameters()) / 1e6, 2),
                gflops=round(fc.get_total_flops() / 1e9, 1), gpu_ms=round(zamanla(), 1),
                cihaz=torch.cuda.get_device_name(0) if cihaz.type == "cuda" else "cpu")

def gorsel(tah, yol):
    """Sınama karolarında örnekler: yeşil = doğru hasar, kırmızı = yanlış alarm, mavi = kaçırılan hasar."""
    def kiou(i):
        p, g = tah[i].astype(bool) & (Y[i] != 255), Y[i] == 1
        u = (p | g).sum()
        return float((p & g).sum() / u) if u else float("nan")
    kp = sorted((kiou(i), i) for i in te if S[i] and KPD[i] and (Y[i] == 1).any())
    sec = []
    if kp:
        sec += [(kp[-1][1], "en iyi"), (kp[len(kp) // 2][1], "ortanca"), (kp[0][1], "en kötü")]
    mb = sorted((kiou(i), i) for i in dm if S[i] and (Y[i] == 1).any())
    if mb:
        sec.append((mb[len(mb) // 2][1], "dış sınama MEDBFS 0,75 m (ortanca)"))
    if dq:
        sec.append((max(dq, key=lambda i: int((tah[i].astype(bool) & (Y[i] == 0)).sum())),
                    "dış sınama QQB (en çok yanlış alarm)"))
        yq = [i for i in dq if (Y[i] == 1).any()]
        if yq:
            sec.append((max(yq, key=lambda i: int((Y[i] == 1).sum())), "dış sınama QQB (en çok yıkık bina)"))
    yh = sorted((kiou(i), i) for i in dh if (Y[i] == 1).any())
    if yh:
        sec.append((yh[len(yh) // 2][1], "dış sınama HOTOSM Adıyaman 0,5 m (ortanca)"))
    ng = [i for i in te if not S[i]]
    if ng:
        sec.append((ng[0], "deprem öncesi (hasarsız)"))
    sec = list(dict(sec).items())  # aynı karo iki kez seçildiyse bir kez
    if not sec:
        return
    fig, ax = plt.subplots(len(sec), 3, figsize=(10.5, 3.6 * len(sec)), squeeze=False)
    for r, (i, ad) in enumerate(sec):
        k = kullan.loc[i]
        o = X[i][..., :3].astype(float) / 255
        p, g = tah[i].astype(bool) & (Y[i] != 255), Y[i] == 1
        renk = np.zeros_like(o); renk[p & g] = (0.1, 0.9, 0.1); renk[p & ~g] = (0.95, 0.1, 0.1); renk[~p & g] = (0.1, 0.45, 1.0)
        ax[r, 0].imshow(X[i][..., :3]); ax[r, 0].set_title(f"{ad}\n{k.kume} · {k.il}/{k.ilce}", fontsize=9)
        ax[r, 1].imshow(X[i][..., :3]); ax[r, 1].imshow(np.ma.masked_where(~g, g.astype(np.uint8)), cmap="autumn", alpha=0.6, vmin=0, vmax=1)
        if (Y[i] == 255).any():
            ax[r, 1].imshow(np.ma.masked_where(Y[i] != 255, np.ones(Y[i].shape)), cmap="Greys", alpha=0.35, vmin=0, vmax=1)
        ax[r, 1].set_title(f"Gerçek etiket (hasarlı piksel %{100 * g.mean():.1f}; gri = yok say)", fontsize=9)
        ax[r, 2].imshow(np.where((p | g)[..., None], 0.3 * o + 0.7 * renk, o))
        t = "hasar yok" if not p.any() else (f"karo IoU {kiou(i):.2f}" if g.any() else f"hasar dediği piksel %{100 * p.mean():.2f}")
        ax[r, 2].set_title(f"{MODEL_ADI}: {t}", fontsize=9)
        for a in ax[r]:
            a.axis("off")
    fig.suptitle("yeşil = doğru bulunan hasar · kırmızı = yanlış alarm · mavi = kaçırılan hasar", fontsize=11, y=1.0)
    fig.tight_layout()
    fig.savefig(yol, dpi=85, pil_kwargs={"quality": 88}, bbox_inches="tight")
    plt.close(fig)
    if display:
        display(IPImage(filename=yol))

for tohum in TOHUMLAR:
    klasor = f"{MODEL_KLASORU}/tohum_{tohum}"
    if os.path.exists(f"{klasor}/sonuc.json"):
        print("tamamlanmış, atlanıyor:", klasor); continue
    os.makedirs(klasor, exist_ok=True)
    t0 = time.time()
    model, egitim = egit(tr, va, tohum, klasor)
    sure = round((time.time() - t0) / 60, 1)
    tah = tahmin_et(model, te + va + dm + dq + dh)
    teS = alt(te, S)
    sonuc = dict(
        model=MODEL_ADI, deney=DENEY, protokol=PROTOKOL, tohum=tohum,
        tarih=datetime.datetime.now().isoformat(timespec="seconds"),
        ayarlar=dict(donem=DONEM, sabir=SABIR, parti=PARTI, birikim=BIRIKIM, ogrenme_orani=OGRENME_ORANI, esik=ESIK,
                     kayip=f"{KAYIP_FOCAL:g} focal(a={FOCAL_ALFA:g},g=2) + {1 - KAYIP_FOCAL:g} dice; 255 yok say",
                     optimizer="AdamW wd=1e-2", cizelge="CosineAnnealingWarmRestarts T0=5 Tmult=2", amp=True, kirpma=1.0,
                     qqb_egitime=QQB_EGITIME, hotosm_egitime=HOTOSM_EGITIME, bolge_egitimi=BOLGE_EGITIMI,
                     focal_alfa=FOCAL_ALFA, ornekleyici=ORNEKLEYICI, girdi=GIRDI, artirim_tohumlu=True, duman_testi=DUMAN_TESTI),
        yazilim=dict(torch=torch.__version__, smp=smp.__version__, python=platform.python_version(),
                     albumentations=A.__version__, timm=__import__("timm").__version__,
                     numpy=np.__version__, ortam="kaggle"),
        maliyet=maliyet_olc(model),
        egitim=dict(en_iyi_donem=egitim["en_iyi_donem"], en_iyi_dogrulama_iou=egitim["en_iyi_dogrulama_iou"],
                    donem_sayisi=egitim["donem_sayisi"], sure_dk=sure,
                    karo=dict(egitim=len(tr), dogrulama=len(va), sinama=len(te), dis_medbfs=len(dm), dis_qqb=len(dq),
                              dis_hotosm=len(dh), egitim_kumeleri={f"{a}/{b}": int(n) for (a, b), n in
                                                                   kullan.loc[tr].groupby(["kume", "goruntu"]).size().items()})),
        gecmis=[[g["donem"], round(g["egitim_kaybi"], 4), round(g["dogrulama_iou"], 4), round(g["dogrulama_f1"], 4)]
                for g in egitim["gecmis"]],
        sinama_sonrasi_tumu=olcutler(tah, teS),
        sinama_katepd=olcutler(tah, alt(teS, KPD)),
        sinama_medbfs=olcutler(tah, alt(teS, (kullan.kume == "MEDBFS").values)),
        sinama_hatay=olcutler(tah, alt(teS, (kullan.il == "Hatay").values)),
        sinama_kahramanmaras=olcutler(tah, alt(teS, (kullan.il == "Kahramanmaraş").values)),
        sinama_maxar_0375=olcutler(tah, alt(teS, KPD & (kullan.kaynak == "maxar").values)),
        sinama_pleiades_05=olcutler(tah, alt(teS, KPD & (kullan.kaynak == "pleiades").values)),
        sinama_negatif=yanlis_alarm(tah, alt(te, ~S)),
        # öncesi görüntüsü olan sınama karoları (KATE-CD çiftleri): değişim tespiti deneyiyle aynı karolarda kıyas
        sinama_cd_ikizli=olcutler(tah, [i for i in teS if kullan.ornek[i] in IKIZLI]),
        dogrulama_cd_ikizli=olcutler(tah, [i for i in alt(va, S) if kullan.ornek[i] in IKIZLI]),
        dogrulama_sonrasi=olcutler(tah, alt(va, S)),
        dis_medbfs=olcutler(tah, alt(dm, S)),
        dis_medbfs_negatif=yanlis_alarm(tah, alt(dm, ~S)),
        dis_qqb=olcutler(tah, dq),
        dis_qqb_bina=qqb_bina(tah, dq),
        dis_hotosm=olcutler(tah, dh),
        dis_hotosm_bina=qqb_bina(tah, dh),
        bina_duzeyi_katepd=bilesenler(tah, alt(teS, KPD)),
        morfoloji={str(s): olcutler(morfoloji(tah, teS, s), teS) for s in (1, 2)},
    )
    json.dump(sonuc, open(f"{klasor}/sonuc.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)
    # Tam sonuç (dönem geçmişi dahil) çıktıya da yazılır: yerel kopya bundan alınır (araclar/kosu_kaydet.py)
    print("===SONUC_JSON_BASLA===")
    print(json.dumps(sonuc, ensure_ascii=False, default=float))
    print("===SONUC_JSON_BITTI===")
    # Aynı sonucun sıkıştırılmış tek satırı (tarayıcıdan parça parça okumak için kısa)
    print("===SONUC_GZ_BASLA===")
    print(base64.b64encode(gzip.compress(json.dumps(sonuc, ensure_ascii=False, default=float).encode("utf-8"), 9)).decode())
    print("===SONUC_GZ_BITTI===")
    # Bütün koşuların tek satırlık özeti: Drive'daki tum_kosular.csv'ye eklenir (yalnız ekleme; silme/üzerine yazma yok)
    g_ = lambda k, a: (sonuc.get(k) or {}).get(a)
    ozet_satiri = dict(deney=DENEY, model=MODEL_ADI, tohum=tohum, tarih=sonuc["tarih"], sure_dk=sure,
                       en_iyi_donem=egitim["en_iyi_donem"], dogrulama_iou=egitim["en_iyi_dogrulama_iou"],
                       egitim_karo=len(tr), sinama_iou=g_("sinama_sonrasi_tumu", "iou_hasar"),
                       sinama_kesinlik=g_("sinama_sonrasi_tumu", "kesinlik"), sinama_duyarlilik=g_("sinama_sonrasi_tumu", "duyarlilik"),
                       sinama_f1=g_("sinama_sonrasi_tumu", "f1"), sinama_iou_arka=g_("sinama_sonrasi_tumu", "iou_arka"),
                       sinama_maxar_iou=g_("sinama_maxar_0375", "iou_hasar"), sinama_pleiades_iou=g_("sinama_pleiades_05", "iou_hasar"),
                       sinama_hatay_iou=g_("sinama_hatay", "iou_hasar"), sinama_maras_iou=g_("sinama_kahramanmaras", "iou_hasar"),
                       negatif_yanlis_piksel=g_("sinama_negatif", "yanlis_piksel_orani"), dis_medbfs_iou=g_("dis_medbfs", "iou_hasar"),
                       qqb_saglam_yanlis_alarm=g_("dis_qqb_bina", "saglam_yanlis_alarm_orani"),
                       qqb_yikik_duyarlilik=g_("dis_qqb_bina", "yikik_duyarlilik"),
                       adiyaman_yikik_duyarlilik=g_("dis_hotosm_bina", "yikik_duyarlilik"),
                       parametre_m=sonuc["maliyet"]["parametre_m"], gflops=sonuc["maliyet"]["gflops"],
                       gpu_ms=sonuc["maliyet"]["gpu_ms"], cihaz=sonuc["maliyet"]["cihaz"])
    tk = f"{KOK_CIKTI}/tum_kosular.csv"
    pd.DataFrame([ozet_satiri]).to_csv(tk, mode="a", header=not os.path.exists(tk), index=False)
    print("tum_kosular.csv'ye eklendi:", tk)
    if GORSEL and tohum == TOHUMLAR[0]:
        gorsel(tah, f"{klasor}/ornek_ciktilar.jpg")
    del model, tah; gc.collect()
    if cihaz.type == "cuda":
        torch.cuda.empty_cache()

In [ ]:
# 7) Koşu B (isteğe bağlı) — KATE-PD resmî bölmesi 675/78/79, yalnız KATE-PD sonrası görüntüleri
if KOSU_B:
    assert BOLGE_EGITIMI is None, "Koşu B bütün illerle koşulur"
    K = ((kullan.kume == "KATE-PD") & (kullan.goruntu == "sonrasi")).values
    trB = [i for i in range(N) if K[i] and kullan.resmi_bolme[i] == "train"]
    vaB = [i for i in range(N) if K[i] and kullan.resmi_bolme[i] == "validation"]
    teB = [i for i in range(N) if K[i] and kullan.resmi_bolme[i] == "test"]
    print("resmî bölme: eğitim", len(trB), "doğrulama", len(vaB), "sınama", len(teB))
    for tohum in TOHUMLAR:
        klasor = f"{KOK_CIKTI}/kosu_B/{MODEL_ADI}/tohum_{tohum}"
        if os.path.exists(f"{klasor}/sonuc.json"):
            print("tamamlanmış, atlanıyor:", klasor); continue
        os.makedirs(klasor, exist_ok=True)
        t0 = time.time()
        model, egitim = egit(trB, vaB, tohum, klasor)
        tah = tahmin_et(model, teB + vaB)
        sonuc = dict(model=MODEL_ADI, tohum=tohum, egitim_suresi_dk=round((time.time() - t0) / 60, 1),
                     en_iyi_donem=egitim["en_iyi_donem"], sinama=olcutler(tah, teB), dogrulama=olcutler(tah, vaB),
                     not_="yayımlanmış taban (U-Net ResNet-152) doğrulama IoU 0,36; onların ölçütü parti başına ortalama")
        json.dump(sonuc, open(f"{klasor}/sonuc.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)
        print(json.dumps(sonuc, ensure_ascii=False, default=float))
        del model, tah; gc.collect()

In [ ]:
# 8) Özet: Drive'daki bütün koşulardan Tablo 13 (bu deney) ve Tablo 15 (bölge deneyleri) + iki şekil
#    Tek başına da çalışır: yalnız 0. ve 0b. hücreyi, sonra bu hücreyi çalıştırın.
import glob, json, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
TABLO15_MODEL = globals().get("TABLO15_MODEL", "segformer_mitb4")   # Tablo 15'te kullanılan model

def yaz(tablo_):
    try:
        return tablo_.to_markdown(index=False)
    except ImportError:  # tabulate yoksa
        return tablo_.to_string(index=False)

def oku(desen):
    satir = []
    for yol in sorted(glob.glob(desen)):
        if "/kosu_B/" in yol.replace("\\", "/"):
            continue  # Koşu B'nin sonuç biçimi farklı (yayın kıyası), ayrı raporlanır
        try:
            s = json.load(open(yol, encoding="utf-8"))
            s["egitim"]["en_iyi_donem"]
        except Exception as e:
            print("okunamadı, atlandı:", yol, e); continue
        t = s.get("sinama_sonrasi_tumu") or {}
        m = s.get("maliyet") or {}
        q = s.get("dis_qqb_bina") or {}
        d = s.get("dis_medbfs") or {}
        h_ = s.get("dis_hotosm_bina") or {}
        satir.append(dict(deney=s.get("deney"), model=s["model"], tohum=s["tohum"],
                          adiyaman_duyarlilik=h_.get("yikik_duyarlilik"), dogruluk=t.get("dogruluk"),
                          kesinlik=t.get("kesinlik"), duyarlilik=t.get("duyarlilik"), f1=t.get("f1"),
                          ozgulluk=t.get("ozgulluk"), iou_hasar=t.get("iou_hasar"),
                          ortalama_iou=(t["iou_hasar"] + t["iou_arka"]) / 2 if t else None,
                          parametre_m=m.get("parametre_m"), gflops=m.get("gflops"), gpu_ms=m.get("gpu_ms"),
                          dis_medbfs_iou=d.get("iou_hasar"), qqb_saglam_yanlis_alarm=q.get("saglam_yanlis_alarm_orani"),
                          en_iyi_donem=s["egitim"]["en_iyi_donem"], sure_dk=s["egitim"]["sure_dk"],
                          hatay=(s.get("sinama_hatay") or {}), maras=(s.get("sinama_kahramanmaras") or {})))
    return pd.DataFrame(satir)

def virgul(x, n=4):
    return "—" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{n}f}".replace(".", ",")

SUTUN = [("dogruluk", "Doğruluk", 4), ("kesinlik", "Kesinlik", 4), ("duyarlilik", "Duyarlılık", 4), ("f1", "F₁", 4),
         ("ozgulluk", "Özgüllük", 4), ("iou_hasar", "IoU_hasar", 4), ("parametre_m", "Parametre (M)", 1),
         ("gflops", "GFLOPs", 1), ("gpu_ms", "Çıkarım (ms, T4)", 1), ("ortalama_iou", "Ortalama IoU", 4),
         ("dis_medbfs_iou", "Dış MEDBFS IoU", 4), ("qqb_saglam_yanlis_alarm", "QQB sağlam bina yanlış alarm", 3),
         ("adiyaman_duyarlilik", "Adıyaman yıkık bina duyarlılığı", 3)]
df = oku(f"{KOK_CIKTI}/{DENEY}/*/tohum_*/sonuc.json")
if len(df):
    satirlar = []
    for model, g in df.groupby("model", sort=False):
        h = {"Mimari": model, "Tohum": len(g)}
        for k, ad, n in SUTUN:
            v = g[k].dropna().astype(float)
            h[ad] = "—" if v.empty else (virgul(v.mean(), n) + (f" ± {virgul(v.std(ddof=1), n)}" if len(v) > 1 else ""))
        satirlar.append(h)
    tablo = pd.DataFrame(satirlar)
    print(f"Tablo 13 — deney {DENEY} (sınama, bütün pikseller; ± = tohumlar arası standart sapma)")
    print(yaz(tablo))
    df.drop(columns=["hatay", "maras"]).to_csv(f"{KOK_CIKTI}/{DENEY}/tablo13_tum_kosular.csv", index=False)
    tablo.to_csv(f"{KOK_CIKTI}/{DENEY}/tablo13_ozet.csv", index=False)
    # Şekiller (300 dpi)
    ozet = df.groupby("model").agg(iou=("iou_hasar", "mean"), sd=("iou_hasar", "std"), gflops=("gflops", "mean"),
                                   param=("parametre_m", "mean")).sort_values("iou")
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(ozet) + 1.2))
    ax.barh(ozet.index, ozet.iou, xerr=ozet.sd.fillna(0), color="#3b6ea5", capsize=3)
    ax.set_xlabel("Sınama IoU_hasar"); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: virgul(v, 2)))
    ax.set_title(f"Mimarilere göre sınama IoU_hasar ({DENEY})"); fig.tight_layout()
    fig.savefig(f"{KOK_CIKTI}/{DENEY}/sekil_iou_modeller.png", dpi=300); plt.close(fig)
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.scatter(ozet.gflops, ozet.iou, s=ozet.param * 4, color="#c0504d", alpha=0.7)
    for m_, r in ozet.iterrows():
        ax.annotate(m_, (r.gflops, r.iou), fontsize=7, xytext=(4, 3), textcoords="offset points")
    ax.set_xlabel("GFLOPs (1 × 3 × 512 × 512)"); ax.set_ylabel("Sınama IoU_hasar")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: virgul(v, 2)))
    ax.set_title("Başarım – hesaplama maliyeti (daire büyüklüğü: parametre)"); fig.tight_layout()
    fig.savefig(f"{KOK_CIKTI}/{DENEY}/sekil_iou_gflops.png", dpi=300); plt.close(fig)
    print("kaydedildi:", f"{KOK_CIKTI}/{DENEY}/", "tablo13_ozet.csv, tablo13_tum_kosular.csv, sekil_iou_modeller.png, sekil_iou_gflops.png")
else:
    print("Bu deney için henüz sonuç yok:", f"{KOK_CIKTI}/{DENEY}")

# Bütün deneylerin bütün koşuları (sonuc.json'lardan yeniden üretilir; kaynak gerçek budur) — yerel kopya için yazdırılır
tum_df = oku(f"{KOK_CIKTI}/*/*/tohum_*/sonuc.json")
if len(tum_df):
    tum_df = tum_df.drop(columns=["hatay", "maras"])
    tum_df.to_csv(f"{KOK_CIKTI}/tum_kosular_sonuc_json.csv", index=False)
    print(f"\nBütün koşular: {len(tum_df)} satır, {tum_df.deney.nunique()} deney → {KOK_CIKTI}/tum_kosular_sonuc_json.csv")
    print("===TUM_KOSULAR_BASLA===")
    print(tum_df.to_csv(index=False))
    print("===TUM_KOSULAR_BITTI===")

# Tablo 15: PROTOKOL_bolge_* ve _haric_* deneyleri (SegFormer). Birleşik satırı: _bolge_hatay_maras varsa o, yoksa bütün iller.
ESAS = PROTOKOL + ("_qqb" if QQB_EGITIME else "") + {None: "", "dik": "_hotdik", "erken": "_hoterken"}[HOTOSM_EGITIME]
bolge = {"Hatay": "_bolge_hatay", "Kahramanmaraş": "_bolge_maras", "Birleşik (Hatay + Kahramanmaraş)": "_bolge_hatay_maras",
         "Hatay hariç bütün iller": "_haric_hatay", "Kahramanmaraş hariç bütün iller": "_haric_maras"}
b15 = {ad: oku(f"{KOK_CIKTI}/{ESAS}{ek}/{TABLO15_MODEL}/tohum_*/sonuc.json") for ad, ek in bolge.items()}
if b15["Birleşik (Hatay + Kahramanmaraş)"].empty:
    b15["Birleşik (bütün iller)"] = oku(f"{KOK_CIKTI}/{ESAS}/{TABLO15_MODEL}/tohum_*/sonuc.json")
    del b15["Birleşik (Hatay + Kahramanmaraş)"]
if not b15["Hatay"].empty or not b15["Kahramanmaraş"].empty:
    def ort(d, il, k):
        v = [s.get(k) for s in d[il] if s]
        return float(np.mean(v)) if v else None
    ref = {"hatay": ort(b15["Hatay"], "hatay", "iou_hasar") if not b15["Hatay"].empty else None,
           "maras": ort(b15["Kahramanmaraş"], "maras", "iou_hasar") if not b15["Kahramanmaraş"].empty else None}
    s15 = []
    for egitim_bolgesi, d in b15.items():
        if d.empty:
            continue
        for il, ad in (("hatay", "Hatay"), ("maras", "Kahramanmaraş")):
            iou = ort(d, il, "iou_hasar")
            referans_mi = egitim_bolgesi == ad
            s15.append({"Eğitim bölgesi": egitim_bolgesi, "Test bölgesi": ad, "Tohum": len(d), "IoU_hasar": virgul(iou),
                        "F₁": virgul(ort(d, il, "f1")), "Kesinlik": virgul(ort(d, il, "kesinlik")),
                        "Duyarlılık": virgul(ort(d, il, "duyarlilik")),
                        "Bölge içine göre fark": "— (referans)" if referans_mi else
                        (virgul(iou - ref[il]) if iou is not None and ref[il] is not None else "—"),
                        "Oran (bölge içine göre)": "—" if referans_mi or iou is None or not ref[il] else virgul(iou / ref[il], 2)})
    print(f"\nTablo 15 — bölge içi ve bölgeler arası ({TABLO15_MODEL}, sınama)")
    print(yaz(pd.DataFrame(s15)))
    pd.DataFrame(s15).to_csv(f"{KOK_CIKTI}/{ESAS}_tablo15.csv", index=False)

In [ ]:
# 9) Kuyruk — ilk koşudan sonra KUYRUK'taki deneyler sırayla (0. hücrenin ayarları + deneyin farkı).
#    Bir deney hata verirse atlanır, kuyruk sürer. Hücre kaynakları üreteçte gömüldü (Kaggle'da defter
#    kendi hücrelerini okuyamaz); hücreleri elle değiştirmeyin, üreteçten yeniden üretin.
import json, time, traceback
_KAYNAK = json.loads('{"ayar": "# 0) Ayarlar — her koşudan önce yalnız bu hücreyi değiştirin\\nMODEL_ADI = \\"segformer_mitb4\\"\\n#   MiT-B4 kodlayıcılı : segformer_mitb4 · unet_mitb4 · fpn_mitb4 · deeplabv3plus_mitb4 · upernet_mitb4\\n#   aynı kapasite      : deeplabv3plus_resnet101 · upernet_convnext_small\\n#   ölçek              : segformer_mitb0 · segformer_mitb2\\n#   literatür tabanı   : unet_resnet18 (Türkiye çalışmalarında yaygın hafif taban; Hacıefendioğlu vd. 2024)\\n#   isteğe bağlı       : unet_resnet152 (KATE-PD yayınındaki taban mimari; KOSU_B = True ile yayın kıyası)\\nTOHUMLAR = [42]              # ilk tur tek tohum; en iyi 3 model için sonra [42, 7, 2026]\\nPROTOKOL = \\"P2\\"              # \\"P2\\" önerilen (MEDBFS yalnız dış sınama, SABIR 20) · \\"P1\\" = 19 Eylül Koşu A\\nQQB_EGITIME = False          # True → QQB\'nin 339 karosu eğitime eklenir (A3 deneyi; önce yalnız SegFormer ile)\\nQQB_DIS_SINAMA = True        # 74 QQB karosunda deprem sonrası sağlam binalarda yanlış alarm ölçülür\\nHOTOSM_EGITIME = None        # None · \\"dik\\" (bakış ≤ 30°) · \\"erken\\" (bütün açılar) → HOTOSM yıkık bina karoları eğitime\\n                             #   (0,5 m; pakette yalnız 13 Şubat\'a kadarki sahneler var)\\nHOTOSM_DIS_SINAMA = True     # Adıyaman merkez (hiç eğitilmeyen ilçe): yıkık binaların kaçı bulunuyor\\nBOLGE_EGITIMI = None         # Tablo 15: \\"Hatay\\", \\"Kahramanmaraş\\", \\"Hatay+Kahramanmaraş\\" ya da il-dışarıda-bırakma\\n                             #   \\"haric:Hatay\\" / \\"haric:Kahramanmaraş\\" (hedef il eğitimde yok); None = bütün iller\\nKAYIP_FOCAL = 0.5            # kayıp = KAYIP_FOCAL × focal + (1 − KAYIP_FOCAL) × dice  (Tablo 21 taraması)\\nFOCAL_ALFA = 0.25            # focal kaybında hasar piksellerinin ağırlığı; 0,25 hasarı arka plandan az tartar (A-19)\\nORNEKLEYICI = None           # None · 0.5 → eğitim partilerinde hasarlı karo payı ~%50 (yerine koyarak örnekleme)\\nGIRDI = \\"sonrasi\\"            # \\"sonrasi\\" (10 model tablosu) · \\"cd_oncesi_sonrasi\\" = değişim tespiti: KATE-CD çiftleri,\\n                             #   öncesi + sonrası 6 kanal · \\"cd_sonrasi\\" = aynı çiftlerin yalnız sonrası (kıyas tabanı)\\nKOSU_B = False               # KATE-PD resmî bölmesi (675/78/79) ile ayrıca eğit: yayımlanmış U-Net ResNet-152 kıyası\\nAGIRLIK_SAKLA = True         # False → model ağırlığı Drive\'a yazılmaz (yer kazanır; sonuçlar yine kaydedilir)\\nGORSEL = True                # ilk tohum için örnek çıktı görseli\\nDONEM = 50\\nPARTI = 8\\nBIRIKIM = 1                  # GPU belleği yetmezse PARTI = 4, BIRIKIM = 2 (etkin parti yine 8; bunu raporda belirtin)\\nOGRENME_ORANI = 2e-4\\nESIK = 0.5\\nKUYRUK = []                  # Kaggle: ilk koşudan (yukarıdaki ayarlar) sonra sırayla koşulacak ek deneyler; her öğe bu\\n                             #   hücrenin ayarlarının üstüne yazılır, ör. [dict(HOTOSM_EGITIME=\\"dik\\"), dict(MODEL_ADI=\\"unet_mitb4\\")]\\nKOSU_PAYI_SA = 2.5           # kuyrukta bir deneye ayrılan en uzun süre; 12 saatlik oturuma sığmayacaksa sonraki oturuma kalır\\n# Bu oturumun ayarları (üreteç --ayar); kuyruktaki her deney bunların üstüne yazılır\\nMODEL_ADI = \\"unet_mitb4\\"\\nKUYRUK = [dict(MODEL_ADI=\\"upernet_mitb4\\"), dict(MODEL_ADI=\\"deeplabv3plus_mitb4\\"), dict(MODEL_ADI=\\"upernet_convnext_small\\"),\\n          dict(MODEL_ADI=\\"segformer_mitb4\\", GIRDI=\\"cd_sonrasi\\"), dict(MODEL_ADI=\\"segformer_mitb4\\", GIRDI=\\"cd_oncesi_sonrasi\\")]", "0b": "# 0b) Türetilen ayarlar ve Drive\\nimport os\\nSABIR = {\\"P1\\": 8, \\"P2\\": 20}[PROTOKOL]\\nif GIRDI != \\"sonrasi\\":  # değişim tespiti deneyi: yalnız öncesi görüntüsü olan karolar (QQB/HOTOSM\'da öncesi yok)\\n    assert PROTOKOL == \\"P2\\" and not BOLGE_EGITIMI, \\"değişim tespiti deneyi P2 ve bütün illerle koşulur\\"\\n    QQB_EGITIME, HOTOSM_EGITIME, QQB_DIS_SINAMA, HOTOSM_DIS_SINAMA = False, None, False, False\\nKANAL = 6 if GIRDI == \\"cd_oncesi_sonrasi\\" else 3\\n_BOLGE_KISA = {None: \\"\\", \\"Hatay\\": \\"_bolge_hatay\\", \\"Kahramanmaraş\\": \\"_bolge_maras\\",\\n               \\"Hatay+Kahramanmaraş\\": \\"_bolge_hatay_maras\\", \\"haric:Hatay\\": \\"_haric_hatay\\",\\n               \\"haric:Kahramanmaraş\\": \\"_haric_maras\\"}\\n# Deney adı ayarlardan türetilir: farklı ayar farklı klasöre yazar, hiçbir sonuç üzerine yazılmaz.\\nDENEY = (PROTOKOL + (\\"_qqb\\" if QQB_EGITIME else \\"\\") + {None: \\"\\", \\"dik\\": \\"_hotdik\\", \\"erken\\": \\"_hoterken\\"}[HOTOSM_EGITIME]\\n         + _BOLGE_KISA[BOLGE_EGITIMI] + (\\"\\" if KAYIP_FOCAL == 0.5 else f\\"_focal{KAYIP_FOCAL:g}\\")\\n         + (\\"\\" if FOCAL_ALFA == 0.25 else f\\"_alfa{FOCAL_ALFA:g}\\")\\n         + (\\"\\" if not ORNEKLEYICI else f\\"_ornek{round(100 * ORNEKLEYICI)}\\")\\n         + {\\"sonrasi\\": \\"\\", \\"cd_sonrasi\\": \\"_cd_sonrasi\\", \\"cd_oncesi_sonrasi\\": \\"_cd_oncesi_sonrasi\\"}[GIRDI])\\nDUMAN_TESTI = os.environ.get(\\"QM_DUMAN\\", \\"0\\") == \\"1\\"\\nGECICI = os.environ.get(\\"QM_GECICI\\", \\"/tmp\\")   # indirilen veri ve ara dosyalar (sürüm çıktısına girmez)\\nVERI = os.environ.get(\\"QM_VERI\\", f\\"{GECICI}/veri\\")\\nif DUMAN_TESTI:\\n    KOK_CIKTI = os.environ.get(\\"QM_CIKTI\\", f\\"{GECICI}/ciktilar\\")\\nelse:\\n    # Kaggle: /kaggle/working oturum sonunda sürümün çıktısı olarak saklanır (en çok 20 GB)\\n    KOK_CIKTI = \\"/kaggle/working/QuakeMind_egitim\\"\\nimport time\\nOTURUM_BASI = globals().get(\\"OTURUM_BASI\\", time.time())   # 12 saatlik oturum sınırı için (kuyruk hücresi)\\nMODEL_KLASORU = f\\"{KOK_CIKTI}/{DENEY}/{MODEL_ADI}\\"\\nos.makedirs(VERI, exist_ok=True); os.makedirs(MODEL_KLASORU, exist_ok=True)\\nprint(\\"deney:\\", DENEY, \\"| model:\\", MODEL_ADI, \\"| tohumlar:\\", TOHUMLAR, \\"| sabır:\\", SABIR, \\"| çıktı:\\", MODEL_KLASORU)", "4": "# 4) Bu deneyde her karonun rolü (egitim / dogrulama / sinama / dis_medbfs / dis_qqb) ve belleğe alma\\nimport numpy as np, pyarrow.parquet as pq, gc\\nfrom PIL import Image\\nfrom skimage.exposure import match_histograms\\n\\nIKIZLI = set(liste.ikiz[liste.kume == \\"KATE-CD\\"])   # öncesi görüntüsü olan (KATE-CD\'de ikizi bulunan) KATE-PD karoları\\ndef rol_ata(r):\\n    if r.bolme == \\"tampon\\":\\n        return \\"\\"\\n    if GIRDI != \\"sonrasi\\":  # değişim tespiti: yalnız öncesi görüntüsü olan sonrası karoları; öncesi girdinin parçası\\n        if r.goruntu != \\"sonrasi\\" or not (r.kume == \\"MEDBFS\\" or r.ornek in IKIZLI):\\n            return \\"\\"\\n    if PROTOKOL == \\"P2\\" and r.kume == \\"MEDBFS\\":\\n        return \\"dis_medbfs\\" if (r.bolme in (\\"val\\", \\"test\\") and r.olcut == 1) else \\"\\"\\n    if r.bolme == \\"train\\":\\n        return \\"egitim\\"\\n    if r.olcut != 1:\\n        return \\"\\"\\n    return \\"dogrulama\\" if r.bolme == \\"val\\" else \\"sinama\\"\\nliste[\\"rol\\"] = [rol_ata(r) for r in liste.itertuples()]\\nqk = pd.DataFrame(qqb[\\"karolar\\"])\\nqq = pd.DataFrame(dict(ornek=\\"qqb/\\" + qk.ornek, kume=\\"QQB\\", goruntu=\\"sonrasi\\", bolme=qk.grup, olcut=1, grup=\\"\\",\\n                       il=\\"Gaziantep\\", ilce=\\"İslahiye\\", resmi_bolme=\\"\\", kaynak=\\"qqb_maxar\\", hasar_orani=\\"\\", lat=\\"\\", lon=\\"\\",\\n                       kaynak_dosya=qk.ornek, etiket_dosya=\\"\\", ikiz=\\"\\"))\\nqq[\\"rol\\"] = [(\\"egitim\\" if QQB_EGITIME else \\"\\") if g == \\"train\\" else (\\"dis_qqb\\" if QQB_DIS_SINAMA else \\"\\") for g in qk.grup]\\nhk = pd.DataFrame(hot[\\"karolar\\"], columns=[\\"ornek\\", \\"grup\\", \\"dik\\", \\"il\\", \\"ilce\\"])\\nhq = pd.DataFrame(dict(ornek=\\"hotosm/\\" + hk.ornek, kume=\\"HOTOSM\\", goruntu=\\"sonrasi\\", bolme=hk.grup, olcut=1, grup=\\"\\",\\n                       il=hk.il, ilce=hk.ilce, resmi_bolme=\\"\\", kaynak=\\"hotosm_maxar\\", hasar_orani=\\"\\", lat=\\"\\", lon=\\"\\",\\n                       kaynak_dosya=hk.ornek, etiket_dosya=\\"\\", ikiz=\\"\\"))\\ndef hot_rol(g, dik):\\n    if g == \\"train\\":\\n        return \\"egitim\\" if HOTOSM_EGITIME == \\"erken\\" or (HOTOSM_EGITIME == \\"dik\\" and str(dik) == \\"1\\") else \\"\\"\\n    return \\"dis_hotosm\\" if (g == \\"dis_sinama\\" and HOTOSM_DIS_SINAMA) else \\"\\"\\nhq[\\"rol\\"] = [hot_rol(g, d) for g, d in zip(hk.grup, hk.dik)]\\ntum = pd.concat([liste, qq, hq], ignore_index=True)\\nif BOLGE_EGITIMI:  # Tablo 15: eğitim ve model seçimi yalnız seçilen illerde; sınama bütün illerde, il il raporlanır\\n    if BOLGE_EGITIMI.startswith(\\"haric:\\"):  # il-dışarıda-bırakma: hedef il eğitimde ve model seçiminde yok (A-20)\\n        kisit = tum.rol.isin([\\"egitim\\", \\"dogrulama\\"]) & tum.il.eq(BOLGE_EGITIMI.split(\\":\\", 1)[1])\\n    else:\\n        kisit = tum.rol.isin([\\"egitim\\", \\"dogrulama\\"]) & ~tum.il.isin(BOLGE_EGITIMI.split(\\"+\\"))\\n    tum.loc[kisit, \\"rol\\"] = \\"\\"\\nkullan = tum[tum.rol != \\"\\"].reset_index(drop=True)\\nif DUMAN_TESTI:  # her gruptan birkaç örnek\\n    kullan = kullan.groupby([\\"rol\\", \\"kume\\", \\"goruntu\\", \\"resmi_bolme\\"]).head(3).reset_index(drop=True)\\nprint(kullan.groupby([\\"rol\\", \\"kume\\", \\"goruntu\\"]).size().unstack(fill_value=0))\\n\\ndef pil(b):\\n    return Image.open(io.BytesIO(b[\\"bytes\\"] if isinstance(b, dict) else b))\\ndef maske(im):\\n    a = np.asarray(im)\\n    if a.ndim == 3:\\n        a = a.max(axis=2)\\n    return (a > 0).astype(np.uint8)\\ndef medbfs_yol(ad):\\n    for p in (f\\"{MEDBFS_KOK}/{ad}\\", f\\"{MEDBFS_KOK}/{ad.replace(\'Turkey/\', \'\', 1)}\\"):\\n        if os.path.exists(p):\\n            return p\\n    raise FileNotFoundError(ad)\\n\\nif kullan.kume.isin([\\"QQB\\", \\"HOTOSM\\"]).any():\\n    import rasterio\\n    from rasterio.enums import Resampling\\n    from rasterio.features import rasterize\\n    from rasterio.transform import from_bounds as donusum_kutudan\\n    from rasterio.windows import from_bounds as pencere_kutudan\\n    from shapely.geometry import box, shape\\n    YIKIK = [shape(g) for g in qqb[\\"yikik\\"]]\\n    SAGLAM = [shape(g) for g in qqb[\\"saglam_dis_sinama\\"]]\\n    BELIRSIZ = [shape(g) for g in qqb.get(\\"belirsiz\\", [])]   # A-18: maskede 255\\n    KARO_Q = {k[\\"ornek\\"]: k for k in qqb[\\"karolar\\"]}\\n    def _rasterle(parcalar, tr, tip):\\n        parcalar = [(g, v) for g, v in parcalar if not g.is_empty]\\n        return (rasterize(parcalar, out_shape=(512, 512), transform=tr, fill=0, dtype=tip) if parcalar\\n                else np.zeros((512, 512), tip))\\n    def qqb_karo(ad, kimlikli=False):\\n        \\"\\"\\"Maxar 0,31 m -> 0,375 m, 512 x 512 (qqb_karola.py ile aynı); maske: yıkık = 1, belirsiz = 255, gerisi 0.\\"\\"\\"\\n        k = KARO_Q[ad]; x0, y0, K = float(k[\\"x0\\"]), float(k[\\"y0\\"]), qqb[\\"kenar_m\\"]\\n        with rasterio.open(f\\"{VERI}/qqb/maxar/{k[\'tif\']}_visual.tif\\") as src:\\n            a = src.read([1, 2, 3], window=pencere_kutudan(x0, y0, x0 + K, y0 + K, src.transform),\\n                         out_shape=(3, 512, 512), resampling=Resampling.average)\\n        kutu = box(x0, y0, x0 + K, y0 + K); tr = donusum_kutudan(x0, y0, x0 + K, y0 + K, 512, 512)\\n        yk = [(g.intersection(kutu), j + 1) for j, g in enumerate(YIKIK) if g.intersects(kutu)]\\n        bl = [(g.intersection(kutu), 255) for g in BELIRSIZ if g.intersects(kutu)]\\n        msk = _rasterle(bl + [(g, 1) for g, _ in yk], tr, \\"uint8\\")   # önce 255, sonra 1 (çakışmada yıkık kazanır)\\n        kimlik = None\\n        if kimlikli:  # bina düzeyi ölçüm için bina kimlik rasterleri (sağlam, yıkık)\\n            sg = [(g.intersection(kutu), j + 1) for j, g in enumerate(SAGLAM) if g.intersects(kutu)]\\n            kimlik = (_rasterle(sg, tr, \\"uint16\\"), _rasterle(yk, tr, \\"uint16\\"))\\n        return np.transpose(a, (1, 2, 0)), msk, kimlik\\n    GDAL_ORTAM = dict(GDAL_DISABLE_READDIR_ON_OPEN=\\"EMPTY_DIR\\", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=\\".tif\\", VSI_CACHE=\\"TRUE\\",\\n                      GDAL_HTTP_MULTIRANGE=\\"YES\\", GDAL_HTTP_MERGE_CONSECUTIVE_RANGES=\\"YES\\", GDAL_HTTP_MAX_RETRY=\\"5\\",\\n                      GDAL_HTTP_RETRY_DELAY=\\"3\\")\\n    KARO_H = {k[\\"ornek\\"]: k for k in hot[\\"karolar\\"]}\\n    YIKIK_H = [shape(g) for g in hot[\\"yikik_dis_sinama\\"]]\\n    def hot_goruntu(ad):\\n        \\"\\"\\"Maxar deprem sonrası, 0,5 m, 512 x 512 (hotosm_karola.py ile aynı pencere); oturum içinde önbelleğe yazılır.\\"\\"\\"\\n        onbellek = f\\"{VERI}/hotosm/{ad}.png\\"\\n        if os.path.exists(onbellek):\\n            return np.asarray(Image.open(onbellek).convert(\\"RGB\\"))\\n        k = KARO_H[ad]; x0, y0, K = float(k[\\"x0\\"]), float(k[\\"y0\\"]), hot[\\"kenar_m\\"]\\n        with rasterio.Env(**GDAL_ORTAM), rasterio.open(\\"/vsicurl/\\" + k[\\"url\\"]) as src:\\n            yontem = Resampling.average if src.res[0] < hot[\\"gsd\\"] else Resampling.bilinear\\n            a = src.read([1, 2, 3], window=pencere_kutudan(x0, y0, x0 + K, y0 + K, src.transform),\\n                         out_shape=(3, 512, 512), resampling=yontem)\\n        img = np.transpose(a, (1, 2, 0))\\n        os.makedirs(f\\"{VERI}/hotosm\\", exist_ok=True)\\n        Image.fromarray(img).save(onbellek)\\n        return img\\n    def hot_karo(ad, kimlikli=False):\\n        \\"\\"\\"Görüntü COG\'dan, maske pakette (yıkık = 1, diğer OSM binaları = 255, gerisi 0).\\"\\"\\"\\n        img = hot_goruntu(ad)\\n        msk = np.asarray(Image.open(io.BytesIO(HOT_MASKE.read(f\\"{ad}.png\\"))))\\n        kimlik = None\\n        if kimlikli:  # yalnız yıkık bina kimlikleri (HOTOSM\'da doğrulanmış sağlam bina listesi yok)\\n            k = KARO_H[ad]; x0, y0, K = float(k[\\"x0\\"]), float(k[\\"y0\\"]), hot[\\"kenar_m\\"]\\n            kutu = box(x0, y0, x0 + K, y0 + K); tr = donusum_kutudan(x0, y0, x0 + K, y0 + K, 512, 512)\\n            yk = [(g.intersection(kutu), j + 1) for j, g in enumerate(YIKIK_H) if g.intersects(kutu)]\\n            kimlik = (np.zeros((512, 512), np.uint16), _rasterle(yk, tr, \\"uint16\\"))\\n        return img, msk, kimlik\\n    gerekli = [r.kaynak_dosya for r in kullan.itertuples()\\n               if r.kume == \\"HOTOSM\\" and not os.path.exists(f\\"{VERI}/hotosm/{r.kaynak_dosya}.png\\")]\\n    if gerekli:  # HOTOSM pencereleri paralel okunur (her karo ~0,2–0,5 MB)\\n        from concurrent.futures import ThreadPoolExecutor\\n        t_ = time.time()\\n        with ThreadPoolExecutor(8) as ex:\\n            list(ex.map(hot_goruntu, gerekli))\\n        print(\\"HOTOSM görüntüleri okundu:\\", len(gerekli), \\"karo,\\", round(time.time() - t_), \\"sn\\")\\n\\nkpd = {}\\nfor sp in (\\"train\\", \\"validation\\", \\"test\\"):\\n    for r in pq.read_table(f\\"{VERI}/kate-pd/{sp}.parquet\\", columns=[\\"name\\", \\"image\\", \\"mask\\"]).to_pylist():\\n        kpd[r[\\"name\\"]] = r\\nkcd = {}\\nfor sp in (\\"train\\", \\"validation\\", \\"test\\"):\\n    for i, r in enumerate(pq.read_table(f\\"{VERI}/kate-cd/{sp}.parquet\\", columns=[\\"pre_image\\"]).to_pylist()):\\n        kcd[f\\"{sp}/{i}\\"] = r[\\"pre_image\\"]\\n\\nN = len(kullan)\\nX = np.zeros((N, 512, 512, KANAL), np.uint8)   # 6 kanal: sonrası RGB + öncesi RGB (değişim tespiti)\\nONCESI_CD = {r.ikiz: r.kaynak_dosya for r in liste.itertuples() if r.kume == \\"KATE-CD\\"}\\nONCESI_MB = {r.ikiz: r.kaynak_dosya for r in liste.itertuples() if r.kume == \\"MEDBFS\\" and r.goruntu == \\"oncesi\\"}\\nY = np.zeros((N, 512, 512), np.uint8)\\nBINA = {}\\nsonrasi_rgb = {}\\nt = time.time()\\nsira = list(kullan.index[kullan.goruntu == \\"sonrasi\\"]) + list(kullan.index[kullan.goruntu == \\"oncesi\\"])\\nfor i in sira:\\n    r = kullan.loc[i]\\n    if r.kume == \\"QQB\\":\\n        img, msk, kimlik = qqb_karo(r.kaynak_dosya, kimlikli=r.rol == \\"dis_qqb\\")\\n        if kimlik is not None:\\n            BINA[i] = kimlik\\n    elif r.kume == \\"HOTOSM\\":\\n        img, msk, kimlik = hot_karo(r.kaynak_dosya, kimlikli=r.rol == \\"dis_hotosm\\")\\n        if kimlik is not None:\\n            BINA[i] = kimlik\\n    elif r.kume == \\"KATE-PD\\":\\n        k = kpd[r.kaynak_dosya]\\n        img = np.asarray(pil(k[\\"image\\"]).convert(\\"RGB\\")); msk = maske(pil(k[\\"mask\\"]))\\n    elif r.kume == \\"MEDBFS\\" and r.goruntu == \\"sonrasi\\":\\n        img = np.asarray(Image.open(medbfs_yol(r.kaynak_dosya)).convert(\\"RGB\\"))\\n        msk = maske(Image.open(medbfs_yol(r.etiket_dosya)))\\n    else:  # öncesi görüntüsü: boş maske, renkleri ikiz sonrası görüntüye uydurulur\\n        if r.kume == \\"KATE-CD\\":\\n            img = np.asarray(pil(kcd[r.kaynak_dosya]).convert(\\"RGB\\"))\\n        else:\\n            img = np.asarray(Image.open(medbfs_yol(r.kaynak_dosya)).convert(\\"RGB\\"))\\n        ikiz = sonrasi_rgb.get(r.ikiz)\\n        if ikiz is None:  # ikiz bu deneyde yüklenmediyse doğrudan oku\\n            if r.ikiz.startswith(\\"katepd/\\"):\\n                ikiz = np.asarray(pil(kpd[r.ikiz.split(\\"/\\", 1)[1]][\\"image\\"]).convert(\\"RGB\\"))\\n            else:\\n                ikiz = np.asarray(Image.open(medbfs_yol(\\"Turkey/Image/Post-disaster/\\" + r.ikiz.split(\\"/\\", 1)[1])).convert(\\"RGB\\"))\\n        img = np.clip(match_histograms(img, ikiz, channel_axis=-1), 0, 255).astype(np.uint8)\\n        msk = np.zeros((512, 512), np.uint8)\\n    assert img.shape == (512, 512, 3) and msk.shape == (512, 512), (r.ornek, img.shape, msk.shape)\\n    X[i, ..., :3] = img; Y[i] = msk\\n    if KANAL == 6:  # öncesi görüntüsü; renkleri sonrasına uydurulur (öncesi negatiflerindeki gibi)\\n        on = (np.asarray(pil(kcd[ONCESI_CD[r.ornek]]).convert(\\"RGB\\")) if r.kume == \\"KATE-PD\\"\\n              else np.asarray(Image.open(medbfs_yol(ONCESI_MB[r.ornek])).convert(\\"RGB\\")))\\n        X[i, ..., 3:] = np.clip(match_histograms(on, img, channel_axis=-1), 0, 255).astype(np.uint8)\\n    if r.goruntu == \\"sonrasi\\":\\n        sonrasi_rgb[r.ornek] = img\\ndel sonrasi_rgb, kpd, kcd\\ngc.collect()\\nprint(N, \\"karo belleğe alındı,\\", round(time.time() - t), \\"sn | bellek ~\\", round((X.nbytes + Y.nbytes) / 1e9, 2), \\"GB\\")", "5": "# 5) Modeller, veri artırımı, ölçütler ve eğitim döngüsü\\nimport random, copy\\nimport torch.nn as nn\\nimport albumentations as A\\nfrom albumentations.pytorch import ToTensorV2\\nfrom scipy import ndimage as ndi\\nfrom torch.utils.data import Dataset, DataLoader\\n\\nMIMARILER = {\\n    \\"segformer_mitb4\\": lambda w: smp.Segformer(encoder_name=\\"mit_b4\\", encoder_weights=w, classes=1, in_channels=KANAL),\\n    \\"unet_mitb4\\": lambda w: smp.Unet(encoder_name=\\"mit_b4\\", encoder_weights=w, classes=1, in_channels=KANAL),\\n    \\"fpn_mitb4\\": lambda w: smp.FPN(encoder_name=\\"mit_b4\\", encoder_weights=w, classes=1, in_channels=KANAL),\\n    \\"deeplabv3plus_mitb4\\": lambda w: smp.DeepLabV3Plus(encoder_name=\\"mit_b4\\", encoder_weights=w, classes=1, in_channels=KANAL),\\n    \\"upernet_mitb4\\": lambda w: smp.UPerNet(encoder_name=\\"mit_b4\\", encoder_weights=w, classes=1, in_channels=KANAL),\\n    \\"deeplabv3plus_resnet101\\": lambda w: smp.DeepLabV3Plus(encoder_name=\\"resnet101\\", encoder_weights=w, classes=1, in_channels=KANAL),\\n    \\"upernet_convnext_small\\": lambda w: smp.UPerNet(encoder_name=\\"tu-convnext_small\\", encoder_weights=w, classes=1, in_channels=KANAL),\\n    \\"segformer_mitb0\\": lambda w: smp.Segformer(encoder_name=\\"mit_b0\\", encoder_weights=w, classes=1, in_channels=KANAL),\\n    \\"segformer_mitb2\\": lambda w: smp.Segformer(encoder_name=\\"mit_b2\\", encoder_weights=w, classes=1, in_channels=KANAL),\\n    \\"unet_resnet18\\": lambda w: smp.Unet(encoder_name=\\"resnet18\\", encoder_weights=w, classes=1, in_channels=KANAL),\\n    \\"unet_resnet152\\": lambda w: smp.Unet(encoder_name=\\"resnet152\\", encoder_weights=w, classes=1, in_channels=KANAL),\\n}\\nassert MODEL_ADI in MIMARILER, \\"MODEL_ADI şunlardan biri olmalı: \\" + \\", \\".join(MIMARILER)\\n\\ndef tohumla(s):\\n    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)\\n    # albumentations 2.x global tohumu dikkate almaz (22 Eylül ölçüldü): artırım ayrıca tohumlanır; DataLoader\\n    # işçileri torch\'un işçi tohumundan (_isci_tohumu) — aynı tohum aynı artırımı, her dönem ve işçi farklı artırımı verir\\n    egitim_donusumu.set_random_seed(s); deger_donusumu.set_random_seed(s)\\n\\ndef _isci_tohumu(_):\\n    w = torch.utils.data.get_worker_info()\\n    w.dataset.d.set_random_seed(torch.initial_seed() % 2**32)\\n\\nORT, SS = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]\\n# değişim tespitinde öncesi görüntüsü aynı geometrik ve renk dönüşümünü alır (additional_targets)\\n_EK = {\\"additional_targets\\": {\\"oncesi\\": \\"image\\"}} if KANAL == 6 else {}\\negitim_donusumu = A.Compose([\\n    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),\\n    A.Affine(translate_percent=(-0.05, 0.05), scale=(0.9, 1.1), rotate=(-20, 20), p=0.5, border_mode=4),\\n    A.RandomResizedCrop(size=(512, 512), scale=(0.7, 1.0), ratio=(0.9, 1.1), p=0.3),\\n    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.5),\\n    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.2),\\n    A.GaussianBlur(blur_limit=(3, 5), p=0.15), A.GaussNoise(std_range=(0.02, 0.1), p=0.15),\\n    A.Normalize(mean=ORT, std=SS), ToTensorV2(),\\n], **_EK)\\ndeger_donusumu = A.Compose([A.Normalize(mean=ORT, std=SS), ToTensorV2()], **_EK)\\n\\nclass Karolar(Dataset):\\n    def __init__(self, idx, donusum):\\n        self.idx, self.d = np.asarray(idx), donusum\\n    def __len__(self):\\n        return len(self.idx)\\n    def __getitem__(self, k):\\n        i = self.idx[k]\\n        if KANAL == 6:\\n            a = self.d(image=np.ascontiguousarray(X[i][..., :3]), oncesi=np.ascontiguousarray(X[i][..., 3:]), mask=Y[i])\\n            return torch.cat([a[\\"image\\"], a[\\"oncesi\\"]], 0), a[\\"mask\\"].float().unsqueeze(0), i\\n        a = self.d(image=X[i], mask=Y[i])\\n        return a[\\"image\\"], a[\\"mask\\"].float().unsqueeze(0), i\\n\\ndef yukleyici(idx, egitim):\\n    ornekleyici = None\\n    if egitim and ORNEKLEYICI:  # hasarlı karo payı ~ORNEKLEYICI olacak şekilde yerine koyarak örnekleme\\n        hasarli = np.array([(Y[i] == 1).any() for i in idx])\\n        w = np.where(hasarli, ORNEKLEYICI / max(int(hasarli.sum()), 1), (1 - ORNEKLEYICI) / max(int((~hasarli).sum()), 1))\\n        ornekleyici = torch.utils.data.WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),\\n                                                             num_samples=len(idx), replacement=True)\\n    return DataLoader(Karolar(idx, egitim_donusumu if egitim else deger_donusumu), batch_size=PARTI,\\n                      shuffle=egitim and ornekleyici is None, sampler=ornekleyici,\\n                      num_workers=0 if DUMAN_TESTI else 2, pin_memory=not DUMAN_TESTI,\\n                      worker_init_fn=None if DUMAN_TESTI else _isci_tohumu,\\n                      drop_last=egitim and len(idx) > PARTI)\\n\\ncihaz = torch.device(\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n\\n@torch.no_grad()\\ndef tahmin_et(model, idx):\\n    \\"\\"\\"Karo başına ikili tahmin maskesi (uint8).\\"\\"\\"\\n    model.eval(); out = {}\\n    for x, _, ii in yukleyici(idx, False):\\n        with torch.autocast(device_type=cihaz.type, dtype=torch.float16, enabled=cihaz.type == \\"cuda\\"):\\n            p = torch.sigmoid(model(x.to(cihaz, non_blocking=True))).float()\\n        for j, i in enumerate(ii.tolist()):\\n            out[i] = (p[j, 0] > ESIK).cpu().numpy().astype(np.uint8)\\n    return out\\n\\ndef olcutler(tah, idx):\\n    \\"\\"\\"Tüm pikseller üzerinden (global) ölçütler + karo başına IoU ortalaması. 255 (yok say) pikseller sayılmaz.\\n    Boş kümede None.\\"\\"\\"\\n    if len(idx) == 0:\\n        return None\\n    tp = fp = fn = tn = 0; karo_iou = []\\n    for i in idx:\\n        v = Y[i] != 255\\n        p, g = tah[i].astype(bool) & v, Y[i] == 1\\n        a, b, c = int((p & g).sum()), int((p & ~g).sum()), int((~p & g).sum())\\n        tp += a; fp += b; fn += c; tn += int((~p & ~g & v).sum())\\n        if g.any() or p.any():\\n            karo_iou.append(a / max(a + b + c, 1))\\n    e = 1e-9\\n    return dict(n_karo=len(idx), iou_hasar=tp / (tp + fp + fn + e), kesinlik=tp / (tp + fp + e),\\n                duyarlilik=tp / (tp + fn + e), f1=2 * tp / (2 * tp + fp + fn + e), ozgulluk=tn / (tn + fp + e),\\n                dogruluk=(tp + tn) / (tp + tn + fp + fn + e), iou_arka=tn / (tn + fp + fn + e),\\n                karo_ortalama_iou=float(np.mean(karo_iou)) if karo_iou else None)\\n\\ndef yanlis_alarm(tah, idx, min_piksel=64):\\n    \\"\\"\\"Hasarsız karolarda: hasar denen piksel oranı ve >= min_piksel hasar denen karo oranı.\\"\\"\\"\\n    if len(idx) == 0:\\n        return None\\n    hasar = {i: tah[i].astype(bool) & (Y[i] != 255) for i in idx}\\n    px = [float(hasar[i].sum() / max(int((Y[i] != 255).sum()), 1)) for i in idx]\\n    return dict(n_karo=len(idx), yanlis_piksel_orani=float(np.mean(px)),\\n                alarm_veren_karo_orani=float(np.mean([hasar[i].sum() >= min_piksel for i in idx])))\\n\\ndef gsd(i):\\n    k = kullan.kaynak[i]\\n    return 0.375 if k in (\\"maxar\\", \\"qqb_maxar\\") else 0.5 if k in (\\"pleiades\\", \\"hotosm_maxar\\") else 0.751\\n\\ndef bilesenler(tah, idx, min_px=10):\\n    \\"\\"\\"Bağlantılı bileşen (≈ bina kümesi) düzeyi: boyuta göre bulunma oranı ve bileşen kesinliği.\\"\\"\\"\\n    if len(idx) == 0:\\n        return None\\n    boy = {\\"kucuk (<50 m2)\\": [0, 0], \\"orta (50-200 m2)\\": [0, 0], \\"buyuk (>200 m2)\\": [0, 0]}\\n    kap25 = top = tah_top = yanlis = 0\\n    for i in idx:\\n        p, g = tah[i].astype(bool) & (Y[i] != 255), Y[i] == 1\\n        lg, ng = ndi.label(g, structure=np.ones((3, 3)))\\n        for k in range(1, ng + 1):\\n            m = lg == k\\n            alan = m.sum() * gsd(i) ** 2\\n            s = \\"kucuk (<50 m2)\\" if alan < 50 else \\"orta (50-200 m2)\\" if alan <= 200 else \\"buyuk (>200 m2)\\"\\n            boy[s][0] += int((p & m).any()); boy[s][1] += 1\\n            kap25 += int((p & m).sum() >= 0.25 * m.sum()); top += 1\\n        lp, npr = ndi.label(p, structure=np.ones((3, 3)))\\n        for k in range(1, npr + 1):\\n            m = lp == k\\n            if m.sum() < min_px:\\n                continue\\n            tah_top += 1; yanlis += int(not (m & g).any())\\n    return dict(boyuta_gore={k: dict(bulunan=v[0], toplam=v[1], oran=(v[0] / v[1] if v[1] else None)) for k, v in boy.items()},\\n                bina_duyarliligi_ortusme=sum(v[0] for v in boy.values()) / max(top, 1),\\n                bina_duyarliligi_yuzde25=kap25 / max(top, 1), gercek_bilesen=top, tahmin_bilesen=tah_top,\\n                yanlis_alarm_bilesen=yanlis, bilesen_kesinligi=(tah_top - yanlis) / max(tah_top, 1))\\n\\ndef qqb_bina(tah, idx, oran=0.25):\\n    \\"\\"\\"QQB dış sınama: sağlam binalarda yanlış alarm, yıkık binalarda bulunma (bina ayak izinin >= %25\'i hasar).\\"\\"\\"\\n    if len(idx) == 0:\\n        return None\\n    sg, yk = {}, {}\\n    for i in idx:\\n        p = tah[i].astype(np.float64).ravel()\\n        for ids, depo in zip(BINA[i], (sg, yk)):\\n            v = ids.ravel()\\n            if not v.any():\\n                continue\\n            toplam = np.bincount(v); isabet = np.bincount(v, weights=p, minlength=len(toplam))\\n            for j in np.nonzero(toplam)[0]:\\n                if j:\\n                    a = depo.setdefault(int(j), [0.0, 0]); a[0] += isabet[j]; a[1] += int(toplam[j])\\n    so = [a[0] / a[1] for a in sg.values()]; yo = [a[0] / a[1] for a in yk.values()]\\n    ort = lambda v: float(np.mean(v)) if v else None\\n    return dict(saglam_bina=len(so), saglam_yanlis_alarm_bina=int(sum(o >= oran for o in so)),\\n                saglam_yanlis_alarm_orani=ort([o >= oran for o in so]), saglam_herhangi_piksel_orani=ort([o > 0 for o in so]),\\n                yikik_bina=len(yo), yikik_bulunan=int(sum(o >= oran for o in yo)), yikik_duyarlilik=ort([o >= oran for o in yo]))\\n\\ndef morfoloji(tah, idx, seviye):\\n    \\"\\"\\"Seviye k: k yinelemeli açma, ardından k yinelemeli kapama (3 x 3). Tablo 21\'deki 0 / 1 / 2.\\"\\"\\"\\n    st = np.ones((3, 3), bool)\\n    return {i: ndi.binary_closing(ndi.binary_opening(tah[i].astype(bool), st, iterations=seviye), st,\\n                                  iterations=seviye).astype(np.uint8) for i in idx}\\n\\ndef egit(egitim_idx, dogrulama_idx, tohum, klasor):\\n    assert len(dogrulama_idx), \\"doğrulama kümesi boş\\"\\n    tohumla(tohum)\\n    model = MIMARILER[MODEL_ADI](None if DUMAN_TESTI else \\"imagenet\\").to(cihaz)\\n    # ignore_index=255: \\"yok say\\" pikselleri kayba girmez (255 olmayan karolarda sonuç öncekiyle aynı)\\n    dice = smp.losses.DiceLoss(mode=\\"binary\\", ignore_index=255)\\n    focal = smp.losses.FocalLoss(mode=\\"binary\\", alpha=FOCAL_ALFA, gamma=2.0, ignore_index=255)\\n    kayip_f = lambda p, t: KAYIP_FOCAL * focal(p, t) + (1 - KAYIP_FOCAL) * dice(p, t)\\n    opt = torch.optim.AdamW(model.parameters(), lr=OGRENME_ORANI, weight_decay=1e-2)\\n    cz = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=5, T_mult=2, eta_min=1e-6)\\n    olcek = torch.amp.GradScaler(\\"cuda\\", enabled=cihaz.type == \\"cuda\\")\\n    en_iyi, en_iyi_donem, bekle, gecmis, en_iyi_durum = -1.0, 0, 0, [], None\\n    agirlik_yolu = (klasor if AGIRLIK_SAKLA else f\\"{GECICI}/agirlik_gecici\\") + \\"/model_en_iyi.pth\\"\\n    os.makedirs(os.path.dirname(agirlik_yolu), exist_ok=True)\\n    yk = yukleyici(egitim_idx, True)\\n    donem_sayisi = 1 if DUMAN_TESTI else DONEM\\n    for d in range(1, donem_sayisi + 1):\\n        model.train(); t0 = time.time(); top = 0.0; n = 0\\n        opt.zero_grad(set_to_none=True)\\n        for k, (x, y, _) in enumerate(yk):\\n            x, y = x.to(cihaz, non_blocking=True), y.to(cihaz, non_blocking=True)\\n            with torch.autocast(device_type=cihaz.type, dtype=torch.float16, enabled=cihaz.type == \\"cuda\\"):\\n                kayip = kayip_f(model(x), y) / BIRIKIM\\n            olcek.scale(kayip).backward()\\n            if (k + 1) % BIRIKIM == 0 or k + 1 == len(yk):\\n                olcek.unscale_(opt)\\n                nn.utils.clip_grad_norm_(model.parameters(), 1.0)\\n                olcek.step(opt); olcek.update(); opt.zero_grad(set_to_none=True)\\n            top += kayip.item() * BIRIKIM; n += 1\\n            if DUMAN_TESTI and k >= 1:\\n                break\\n        cz.step()\\n        dg = olcutler(tahmin_et(model, dogrulama_idx), dogrulama_idx)\\n        gecmis.append(dict(donem=d, egitim_kaybi=top / max(n, 1), dogrulama_iou=dg[\\"iou_hasar\\"], dogrulama_f1=dg[\\"f1\\"],\\n                           ogrenme_orani=opt.param_groups[0][\\"lr\\"], sure_sn=round(time.time() - t0, 1)))\\n        iyilesti = dg[\\"iou_hasar\\"] > en_iyi\\n        if iyilesti:\\n            en_iyi, en_iyi_donem, bekle = dg[\\"iou_hasar\\"], d, 0\\n            en_iyi_durum = copy.deepcopy(model.state_dict())\\n            torch.save(en_iyi_durum, agirlik_yolu)\\n        else:\\n            bekle += 1\\n        print(f\\"[{MODEL_ADI} t{tohum}] dönem {d}/{donem_sayisi} kayıp {top / max(n, 1):.4f} | doğrulama IoU_hasar \\"\\n              f\\"{dg[\'iou_hasar\']:.4f} F1 {dg[\'f1\']:.4f} | {gecmis[-1][\'sure_sn\']:.0f} sn\\" + (\\" *\\" if iyilesti else \\"\\"), flush=True)\\n        json.dump(gecmis, open(f\\"{klasor}/gecmis.json\\", \\"w\\", encoding=\\"utf-8\\"))\\n        if bekle >= SABIR:\\n            print(f\\"erken durdurma; en iyi dönem {en_iyi_donem} (IoU {en_iyi:.4f})\\"); break\\n    model.load_state_dict(en_iyi_durum)\\n    return model, dict(en_iyi_donem=en_iyi_donem, en_iyi_dogrulama_iou=en_iyi, donem_sayisi=len(gecmis), gecmis=gecmis)\\nprint(\\"hazır\\")", "6": "# 6) Eğitim ve değerlendirme — her tohum için\\nimport datetime, platform\\nimport matplotlib.pyplot as plt\\nfrom torch.utils.flop_counter import FlopCounterMode\\ntry:\\n    from IPython.display import Image as IPImage, display\\nexcept ImportError:  # yerel duman testi\\n    IPImage = display = None\\n\\nsecim = lambda r: kullan.index[kullan.rol == r].tolist()\\ntr, va, te, dm, dq = secim(\\"egitim\\"), secim(\\"dogrulama\\"), secim(\\"sinama\\"), secim(\\"dis_medbfs\\"), secim(\\"dis_qqb\\")\\ndh = secim(\\"dis_hotosm\\")\\nS = (kullan.goruntu == \\"sonrasi\\").values\\nKPD = (kullan.kume == \\"KATE-PD\\").values\\nalt = lambda idx, m: [i for i in idx if m[i]]\\nprint(\\"eğitim\\", len(tr), \\"| doğrulama\\", len(va), \\"| sınama\\", len(te), \\"| dış MEDBFS\\", len(dm), \\"| dış QQB\\", len(dq),\\n      \\"| dış HOTOSM (Adıyaman)\\", len(dh))\\nprint(\\"eğitimdeki kümeler:\\", kullan.loc[tr].groupby([\\"kume\\", \\"goruntu\\"]).size().to_dict())\\n\\ndef maliyet_olc(m):\\n    m = m.eval()\\n    x = torch.randn(1, KANAL, 512, 512, device=cihaz)\\n    with torch.no_grad(), FlopCounterMode(display=False) as fc:\\n        m(x)\\n    def zamanla(n=3 if DUMAN_TESTI else 50):\\n        with torch.no_grad():\\n            for _ in range(3): m(x)\\n            if cihaz.type == \\"cuda\\": torch.cuda.synchronize()\\n            t = time.time()\\n            for _ in range(n): m(x)\\n            if cihaz.type == \\"cuda\\": torch.cuda.synchronize()\\n        return (time.time() - t) / n * 1000\\n    return dict(parametre_m=round(sum(p.numel() for p in m.parameters()) / 1e6, 2),\\n                gflops=round(fc.get_total_flops() / 1e9, 1), gpu_ms=round(zamanla(), 1),\\n                cihaz=torch.cuda.get_device_name(0) if cihaz.type == \\"cuda\\" else \\"cpu\\")\\n\\ndef gorsel(tah, yol):\\n    \\"\\"\\"Sınama karolarında örnekler: yeşil = doğru hasar, kırmızı = yanlış alarm, mavi = kaçırılan hasar.\\"\\"\\"\\n    def kiou(i):\\n        p, g = tah[i].astype(bool) & (Y[i] != 255), Y[i] == 1\\n        u = (p | g).sum()\\n        return float((p & g).sum() / u) if u else float(\\"nan\\")\\n    kp = sorted((kiou(i), i) for i in te if S[i] and KPD[i] and (Y[i] == 1).any())\\n    sec = []\\n    if kp:\\n        sec += [(kp[-1][1], \\"en iyi\\"), (kp[len(kp) // 2][1], \\"ortanca\\"), (kp[0][1], \\"en kötü\\")]\\n    mb = sorted((kiou(i), i) for i in dm if S[i] and (Y[i] == 1).any())\\n    if mb:\\n        sec.append((mb[len(mb) // 2][1], \\"dış sınama MEDBFS 0,75 m (ortanca)\\"))\\n    if dq:\\n        sec.append((max(dq, key=lambda i: int((tah[i].astype(bool) & (Y[i] == 0)).sum())),\\n                    \\"dış sınama QQB (en çok yanlış alarm)\\"))\\n        yq = [i for i in dq if (Y[i] == 1).any()]\\n        if yq:\\n            sec.append((max(yq, key=lambda i: int((Y[i] == 1).sum())), \\"dış sınama QQB (en çok yıkık bina)\\"))\\n    yh = sorted((kiou(i), i) for i in dh if (Y[i] == 1).any())\\n    if yh:\\n        sec.append((yh[len(yh) // 2][1], \\"dış sınama HOTOSM Adıyaman 0,5 m (ortanca)\\"))\\n    ng = [i for i in te if not S[i]]\\n    if ng:\\n        sec.append((ng[0], \\"deprem öncesi (hasarsız)\\"))\\n    sec = list(dict(sec).items())  # aynı karo iki kez seçildiyse bir kez\\n    if not sec:\\n        return\\n    fig, ax = plt.subplots(len(sec), 3, figsize=(10.5, 3.6 * len(sec)), squeeze=False)\\n    for r, (i, ad) in enumerate(sec):\\n        k = kullan.loc[i]\\n        o = X[i][..., :3].astype(float) / 255\\n        p, g = tah[i].astype(bool) & (Y[i] != 255), Y[i] == 1\\n        renk = np.zeros_like(o); renk[p & g] = (0.1, 0.9, 0.1); renk[p & ~g] = (0.95, 0.1, 0.1); renk[~p & g] = (0.1, 0.45, 1.0)\\n        ax[r, 0].imshow(X[i][..., :3]); ax[r, 0].set_title(f\\"{ad}\\\\n{k.kume} · {k.il}/{k.ilce}\\", fontsize=9)\\n        ax[r, 1].imshow(X[i][..., :3]); ax[r, 1].imshow(np.ma.masked_where(~g, g.astype(np.uint8)), cmap=\\"autumn\\", alpha=0.6, vmin=0, vmax=1)\\n        if (Y[i] == 255).any():\\n            ax[r, 1].imshow(np.ma.masked_where(Y[i] != 255, np.ones(Y[i].shape)), cmap=\\"Greys\\", alpha=0.35, vmin=0, vmax=1)\\n        ax[r, 1].set_title(f\\"Gerçek etiket (hasarlı piksel %{100 * g.mean():.1f}; gri = yok say)\\", fontsize=9)\\n        ax[r, 2].imshow(np.where((p | g)[..., None], 0.3 * o + 0.7 * renk, o))\\n        t = \\"hasar yok\\" if not p.any() else (f\\"karo IoU {kiou(i):.2f}\\" if g.any() else f\\"hasar dediği piksel %{100 * p.mean():.2f}\\")\\n        ax[r, 2].set_title(f\\"{MODEL_ADI}: {t}\\", fontsize=9)\\n        for a in ax[r]:\\n            a.axis(\\"off\\")\\n    fig.suptitle(\\"yeşil = doğru bulunan hasar · kırmızı = yanlış alarm · mavi = kaçırılan hasar\\", fontsize=11, y=1.0)\\n    fig.tight_layout()\\n    fig.savefig(yol, dpi=85, pil_kwargs={\\"quality\\": 88}, bbox_inches=\\"tight\\")\\n    plt.close(fig)\\n    if display:\\n        display(IPImage(filename=yol))\\n\\nfor tohum in TOHUMLAR:\\n    klasor = f\\"{MODEL_KLASORU}/tohum_{tohum}\\"\\n    if os.path.exists(f\\"{klasor}/sonuc.json\\"):\\n        print(\\"tamamlanmış, atlanıyor:\\", klasor); continue\\n    os.makedirs(klasor, exist_ok=True)\\n    t0 = time.time()\\n    model, egitim = egit(tr, va, tohum, klasor)\\n    sure = round((time.time() - t0) / 60, 1)\\n    tah = tahmin_et(model, te + va + dm + dq + dh)\\n    teS = alt(te, S)\\n    sonuc = dict(\\n        model=MODEL_ADI, deney=DENEY, protokol=PROTOKOL, tohum=tohum,\\n        tarih=datetime.datetime.now().isoformat(timespec=\\"seconds\\"),\\n        ayarlar=dict(donem=DONEM, sabir=SABIR, parti=PARTI, birikim=BIRIKIM, ogrenme_orani=OGRENME_ORANI, esik=ESIK,\\n                     kayip=f\\"{KAYIP_FOCAL:g} focal(a={FOCAL_ALFA:g},g=2) + {1 - KAYIP_FOCAL:g} dice; 255 yok say\\",\\n                     optimizer=\\"AdamW wd=1e-2\\", cizelge=\\"CosineAnnealingWarmRestarts T0=5 Tmult=2\\", amp=True, kirpma=1.0,\\n                     qqb_egitime=QQB_EGITIME, hotosm_egitime=HOTOSM_EGITIME, bolge_egitimi=BOLGE_EGITIMI,\\n                     focal_alfa=FOCAL_ALFA, ornekleyici=ORNEKLEYICI, girdi=GIRDI, artirim_tohumlu=True, duman_testi=DUMAN_TESTI),\\n        yazilim=dict(torch=torch.__version__, smp=smp.__version__, python=platform.python_version(),\\n                     albumentations=A.__version__, timm=__import__(\\"timm\\").__version__,\\n                     numpy=np.__version__, ortam=\\"kaggle\\"),\\n        maliyet=maliyet_olc(model),\\n        egitim=dict(en_iyi_donem=egitim[\\"en_iyi_donem\\"], en_iyi_dogrulama_iou=egitim[\\"en_iyi_dogrulama_iou\\"],\\n                    donem_sayisi=egitim[\\"donem_sayisi\\"], sure_dk=sure,\\n                    karo=dict(egitim=len(tr), dogrulama=len(va), sinama=len(te), dis_medbfs=len(dm), dis_qqb=len(dq),\\n                              dis_hotosm=len(dh), egitim_kumeleri={f\\"{a}/{b}\\": int(n) for (a, b), n in\\n                                                                   kullan.loc[tr].groupby([\\"kume\\", \\"goruntu\\"]).size().items()})),\\n        gecmis=[[g[\\"donem\\"], round(g[\\"egitim_kaybi\\"], 4), round(g[\\"dogrulama_iou\\"], 4), round(g[\\"dogrulama_f1\\"], 4)]\\n                for g in egitim[\\"gecmis\\"]],\\n        sinama_sonrasi_tumu=olcutler(tah, teS),\\n        sinama_katepd=olcutler(tah, alt(teS, KPD)),\\n        sinama_medbfs=olcutler(tah, alt(teS, (kullan.kume == \\"MEDBFS\\").values)),\\n        sinama_hatay=olcutler(tah, alt(teS, (kullan.il == \\"Hatay\\").values)),\\n        sinama_kahramanmaras=olcutler(tah, alt(teS, (kullan.il == \\"Kahramanmaraş\\").values)),\\n        sinama_maxar_0375=olcutler(tah, alt(teS, KPD & (kullan.kaynak == \\"maxar\\").values)),\\n        sinama_pleiades_05=olcutler(tah, alt(teS, KPD & (kullan.kaynak == \\"pleiades\\").values)),\\n        sinama_negatif=yanlis_alarm(tah, alt(te, ~S)),\\n        # öncesi görüntüsü olan sınama karoları (KATE-CD çiftleri): değişim tespiti deneyiyle aynı karolarda kıyas\\n        sinama_cd_ikizli=olcutler(tah, [i for i in teS if kullan.ornek[i] in IKIZLI]),\\n        dogrulama_cd_ikizli=olcutler(tah, [i for i in alt(va, S) if kullan.ornek[i] in IKIZLI]),\\n        dogrulama_sonrasi=olcutler(tah, alt(va, S)),\\n        dis_medbfs=olcutler(tah, alt(dm, S)),\\n        dis_medbfs_negatif=yanlis_alarm(tah, alt(dm, ~S)),\\n        dis_qqb=olcutler(tah, dq),\\n        dis_qqb_bina=qqb_bina(tah, dq),\\n        dis_hotosm=olcutler(tah, dh),\\n        dis_hotosm_bina=qqb_bina(tah, dh),\\n        bina_duzeyi_katepd=bilesenler(tah, alt(teS, KPD)),\\n        morfoloji={str(s): olcutler(morfoloji(tah, teS, s), teS) for s in (1, 2)},\\n    )\\n    json.dump(sonuc, open(f\\"{klasor}/sonuc.json\\", \\"w\\", encoding=\\"utf-8\\"), ensure_ascii=False, indent=1, default=float)\\n    # Tam sonuç (dönem geçmişi dahil) çıktıya da yazılır: yerel kopya bundan alınır (araclar/kosu_kaydet.py)\\n    print(\\"===SONUC_JSON_BASLA===\\")\\n    print(json.dumps(sonuc, ensure_ascii=False, default=float))\\n    print(\\"===SONUC_JSON_BITTI===\\")\\n    # Aynı sonucun sıkıştırılmış tek satırı (tarayıcıdan parça parça okumak için kısa)\\n    print(\\"===SONUC_GZ_BASLA===\\")\\n    print(base64.b64encode(gzip.compress(json.dumps(sonuc, ensure_ascii=False, default=float).encode(\\"utf-8\\"), 9)).decode())\\n    print(\\"===SONUC_GZ_BITTI===\\")\\n    # Bütün koşuların tek satırlık özeti: Drive\'daki tum_kosular.csv\'ye eklenir (yalnız ekleme; silme/üzerine yazma yok)\\n    g_ = lambda k, a: (sonuc.get(k) or {}).get(a)\\n    ozet_satiri = dict(deney=DENEY, model=MODEL_ADI, tohum=tohum, tarih=sonuc[\\"tarih\\"], sure_dk=sure,\\n                       en_iyi_donem=egitim[\\"en_iyi_donem\\"], dogrulama_iou=egitim[\\"en_iyi_dogrulama_iou\\"],\\n                       egitim_karo=len(tr), sinama_iou=g_(\\"sinama_sonrasi_tumu\\", \\"iou_hasar\\"),\\n                       sinama_kesinlik=g_(\\"sinama_sonrasi_tumu\\", \\"kesinlik\\"), sinama_duyarlilik=g_(\\"sinama_sonrasi_tumu\\", \\"duyarlilik\\"),\\n                       sinama_f1=g_(\\"sinama_sonrasi_tumu\\", \\"f1\\"), sinama_iou_arka=g_(\\"sinama_sonrasi_tumu\\", \\"iou_arka\\"),\\n                       sinama_maxar_iou=g_(\\"sinama_maxar_0375\\", \\"iou_hasar\\"), sinama_pleiades_iou=g_(\\"sinama_pleiades_05\\", \\"iou_hasar\\"),\\n                       sinama_hatay_iou=g_(\\"sinama_hatay\\", \\"iou_hasar\\"), sinama_maras_iou=g_(\\"sinama_kahramanmaras\\", \\"iou_hasar\\"),\\n                       negatif_yanlis_piksel=g_(\\"sinama_negatif\\", \\"yanlis_piksel_orani\\"), dis_medbfs_iou=g_(\\"dis_medbfs\\", \\"iou_hasar\\"),\\n                       qqb_saglam_yanlis_alarm=g_(\\"dis_qqb_bina\\", \\"saglam_yanlis_alarm_orani\\"),\\n                       qqb_yikik_duyarlilik=g_(\\"dis_qqb_bina\\", \\"yikik_duyarlilik\\"),\\n                       adiyaman_yikik_duyarlilik=g_(\\"dis_hotosm_bina\\", \\"yikik_duyarlilik\\"),\\n                       parametre_m=sonuc[\\"maliyet\\"][\\"parametre_m\\"], gflops=sonuc[\\"maliyet\\"][\\"gflops\\"],\\n                       gpu_ms=sonuc[\\"maliyet\\"][\\"gpu_ms\\"], cihaz=sonuc[\\"maliyet\\"][\\"cihaz\\"])\\n    tk = f\\"{KOK_CIKTI}/tum_kosular.csv\\"\\n    pd.DataFrame([ozet_satiri]).to_csv(tk, mode=\\"a\\", header=not os.path.exists(tk), index=False)\\n    print(\\"tum_kosular.csv\'ye eklendi:\\", tk)\\n    if GORSEL and tohum == TOHUMLAR[0]:\\n        gorsel(tah, f\\"{klasor}/ornek_ciktilar.jpg\\")\\n    del model, tah; gc.collect()\\n    if cihaz.type == \\"cuda\\":\\n        torch.cuda.empty_cache()", "8": "# 8) Özet: Drive\'daki bütün koşulardan Tablo 13 (bu deney) ve Tablo 15 (bölge deneyleri) + iki şekil\\n#    Tek başına da çalışır: yalnız 0. ve 0b. hücreyi, sonra bu hücreyi çalıştırın.\\nimport glob, json, os\\nimport numpy as np, pandas as pd\\nimport matplotlib.pyplot as plt\\nfrom matplotlib.ticker import FuncFormatter\\nTABLO15_MODEL = globals().get(\\"TABLO15_MODEL\\", \\"segformer_mitb4\\")   # Tablo 15\'te kullanılan model\\n\\ndef yaz(tablo_):\\n    try:\\n        return tablo_.to_markdown(index=False)\\n    except ImportError:  # tabulate yoksa\\n        return tablo_.to_string(index=False)\\n\\ndef oku(desen):\\n    satir = []\\n    for yol in sorted(glob.glob(desen)):\\n        if \\"/kosu_B/\\" in yol.replace(\\"\\\\\\\\\\", \\"/\\"):\\n            continue  # Koşu B\'nin sonuç biçimi farklı (yayın kıyası), ayrı raporlanır\\n        try:\\n            s = json.load(open(yol, encoding=\\"utf-8\\"))\\n            s[\\"egitim\\"][\\"en_iyi_donem\\"]\\n        except Exception as e:\\n            print(\\"okunamadı, atlandı:\\", yol, e); continue\\n        t = s.get(\\"sinama_sonrasi_tumu\\") or {}\\n        m = s.get(\\"maliyet\\") or {}\\n        q = s.get(\\"dis_qqb_bina\\") or {}\\n        d = s.get(\\"dis_medbfs\\") or {}\\n        h_ = s.get(\\"dis_hotosm_bina\\") or {}\\n        satir.append(dict(deney=s.get(\\"deney\\"), model=s[\\"model\\"], tohum=s[\\"tohum\\"],\\n                          adiyaman_duyarlilik=h_.get(\\"yikik_duyarlilik\\"), dogruluk=t.get(\\"dogruluk\\"),\\n                          kesinlik=t.get(\\"kesinlik\\"), duyarlilik=t.get(\\"duyarlilik\\"), f1=t.get(\\"f1\\"),\\n                          ozgulluk=t.get(\\"ozgulluk\\"), iou_hasar=t.get(\\"iou_hasar\\"),\\n                          ortalama_iou=(t[\\"iou_hasar\\"] + t[\\"iou_arka\\"]) / 2 if t else None,\\n                          parametre_m=m.get(\\"parametre_m\\"), gflops=m.get(\\"gflops\\"), gpu_ms=m.get(\\"gpu_ms\\"),\\n                          dis_medbfs_iou=d.get(\\"iou_hasar\\"), qqb_saglam_yanlis_alarm=q.get(\\"saglam_yanlis_alarm_orani\\"),\\n                          en_iyi_donem=s[\\"egitim\\"][\\"en_iyi_donem\\"], sure_dk=s[\\"egitim\\"][\\"sure_dk\\"],\\n                          hatay=(s.get(\\"sinama_hatay\\") or {}), maras=(s.get(\\"sinama_kahramanmaras\\") or {})))\\n    return pd.DataFrame(satir)\\n\\ndef virgul(x, n=4):\\n    return \\"—\\" if x is None or (isinstance(x, float) and np.isnan(x)) else f\\"{x:.{n}f}\\".replace(\\".\\", \\",\\")\\n\\nSUTUN = [(\\"dogruluk\\", \\"Doğruluk\\", 4), (\\"kesinlik\\", \\"Kesinlik\\", 4), (\\"duyarlilik\\", \\"Duyarlılık\\", 4), (\\"f1\\", \\"F₁\\", 4),\\n         (\\"ozgulluk\\", \\"Özgüllük\\", 4), (\\"iou_hasar\\", \\"IoU_hasar\\", 4), (\\"parametre_m\\", \\"Parametre (M)\\", 1),\\n         (\\"gflops\\", \\"GFLOPs\\", 1), (\\"gpu_ms\\", \\"Çıkarım (ms, T4)\\", 1), (\\"ortalama_iou\\", \\"Ortalama IoU\\", 4),\\n         (\\"dis_medbfs_iou\\", \\"Dış MEDBFS IoU\\", 4), (\\"qqb_saglam_yanlis_alarm\\", \\"QQB sağlam bina yanlış alarm\\", 3),\\n         (\\"adiyaman_duyarlilik\\", \\"Adıyaman yıkık bina duyarlılığı\\", 3)]\\ndf = oku(f\\"{KOK_CIKTI}/{DENEY}/*/tohum_*/sonuc.json\\")\\nif len(df):\\n    satirlar = []\\n    for model, g in df.groupby(\\"model\\", sort=False):\\n        h = {\\"Mimari\\": model, \\"Tohum\\": len(g)}\\n        for k, ad, n in SUTUN:\\n            v = g[k].dropna().astype(float)\\n            h[ad] = \\"—\\" if v.empty else (virgul(v.mean(), n) + (f\\" ± {virgul(v.std(ddof=1), n)}\\" if len(v) > 1 else \\"\\"))\\n        satirlar.append(h)\\n    tablo = pd.DataFrame(satirlar)\\n    print(f\\"Tablo 13 — deney {DENEY} (sınama, bütün pikseller; ± = tohumlar arası standart sapma)\\")\\n    print(yaz(tablo))\\n    df.drop(columns=[\\"hatay\\", \\"maras\\"]).to_csv(f\\"{KOK_CIKTI}/{DENEY}/tablo13_tum_kosular.csv\\", index=False)\\n    tablo.to_csv(f\\"{KOK_CIKTI}/{DENEY}/tablo13_ozet.csv\\", index=False)\\n    # Şekiller (300 dpi)\\n    ozet = df.groupby(\\"model\\").agg(iou=(\\"iou_hasar\\", \\"mean\\"), sd=(\\"iou_hasar\\", \\"std\\"), gflops=(\\"gflops\\", \\"mean\\"),\\n                                   param=(\\"parametre_m\\", \\"mean\\")).sort_values(\\"iou\\")\\n    fig, ax = plt.subplots(figsize=(7, 0.45 * len(ozet) + 1.2))\\n    ax.barh(ozet.index, ozet.iou, xerr=ozet.sd.fillna(0), color=\\"#3b6ea5\\", capsize=3)\\n    ax.set_xlabel(\\"Sınama IoU_hasar\\"); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: virgul(v, 2)))\\n    ax.set_title(f\\"Mimarilere göre sınama IoU_hasar ({DENEY})\\"); fig.tight_layout()\\n    fig.savefig(f\\"{KOK_CIKTI}/{DENEY}/sekil_iou_modeller.png\\", dpi=300); plt.close(fig)\\n    fig, ax = plt.subplots(figsize=(6.5, 4.5))\\n    ax.scatter(ozet.gflops, ozet.iou, s=ozet.param * 4, color=\\"#c0504d\\", alpha=0.7)\\n    for m_, r in ozet.iterrows():\\n        ax.annotate(m_, (r.gflops, r.iou), fontsize=7, xytext=(4, 3), textcoords=\\"offset points\\")\\n    ax.set_xlabel(\\"GFLOPs (1 × 3 × 512 × 512)\\"); ax.set_ylabel(\\"Sınama IoU_hasar\\")\\n    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: virgul(v, 2)))\\n    ax.set_title(\\"Başarım – hesaplama maliyeti (daire büyüklüğü: parametre)\\"); fig.tight_layout()\\n    fig.savefig(f\\"{KOK_CIKTI}/{DENEY}/sekil_iou_gflops.png\\", dpi=300); plt.close(fig)\\n    print(\\"kaydedildi:\\", f\\"{KOK_CIKTI}/{DENEY}/\\", \\"tablo13_ozet.csv, tablo13_tum_kosular.csv, sekil_iou_modeller.png, sekil_iou_gflops.png\\")\\nelse:\\n    print(\\"Bu deney için henüz sonuç yok:\\", f\\"{KOK_CIKTI}/{DENEY}\\")\\n\\n# Bütün deneylerin bütün koşuları (sonuc.json\'lardan yeniden üretilir; kaynak gerçek budur) — yerel kopya için yazdırılır\\ntum_df = oku(f\\"{KOK_CIKTI}/*/*/tohum_*/sonuc.json\\")\\nif len(tum_df):\\n    tum_df = tum_df.drop(columns=[\\"hatay\\", \\"maras\\"])\\n    tum_df.to_csv(f\\"{KOK_CIKTI}/tum_kosular_sonuc_json.csv\\", index=False)\\n    print(f\\"\\\\nBütün koşular: {len(tum_df)} satır, {tum_df.deney.nunique()} deney → {KOK_CIKTI}/tum_kosular_sonuc_json.csv\\")\\n    print(\\"===TUM_KOSULAR_BASLA===\\")\\n    print(tum_df.to_csv(index=False))\\n    print(\\"===TUM_KOSULAR_BITTI===\\")\\n\\n# Tablo 15: PROTOKOL_bolge_* ve _haric_* deneyleri (SegFormer). Birleşik satırı: _bolge_hatay_maras varsa o, yoksa bütün iller.\\nESAS = PROTOKOL + (\\"_qqb\\" if QQB_EGITIME else \\"\\") + {None: \\"\\", \\"dik\\": \\"_hotdik\\", \\"erken\\": \\"_hoterken\\"}[HOTOSM_EGITIME]\\nbolge = {\\"Hatay\\": \\"_bolge_hatay\\", \\"Kahramanmaraş\\": \\"_bolge_maras\\", \\"Birleşik (Hatay + Kahramanmaraş)\\": \\"_bolge_hatay_maras\\",\\n         \\"Hatay hariç bütün iller\\": \\"_haric_hatay\\", \\"Kahramanmaraş hariç bütün iller\\": \\"_haric_maras\\"}\\nb15 = {ad: oku(f\\"{KOK_CIKTI}/{ESAS}{ek}/{TABLO15_MODEL}/tohum_*/sonuc.json\\") for ad, ek in bolge.items()}\\nif b15[\\"Birleşik (Hatay + Kahramanmaraş)\\"].empty:\\n    b15[\\"Birleşik (bütün iller)\\"] = oku(f\\"{KOK_CIKTI}/{ESAS}/{TABLO15_MODEL}/tohum_*/sonuc.json\\")\\n    del b15[\\"Birleşik (Hatay + Kahramanmaraş)\\"]\\nif not b15[\\"Hatay\\"].empty or not b15[\\"Kahramanmaraş\\"].empty:\\n    def ort(d, il, k):\\n        v = [s.get(k) for s in d[il] if s]\\n        return float(np.mean(v)) if v else None\\n    ref = {\\"hatay\\": ort(b15[\\"Hatay\\"], \\"hatay\\", \\"iou_hasar\\") if not b15[\\"Hatay\\"].empty else None,\\n           \\"maras\\": ort(b15[\\"Kahramanmaraş\\"], \\"maras\\", \\"iou_hasar\\") if not b15[\\"Kahramanmaraş\\"].empty else None}\\n    s15 = []\\n    for egitim_bolgesi, d in b15.items():\\n        if d.empty:\\n            continue\\n        for il, ad in ((\\"hatay\\", \\"Hatay\\"), (\\"maras\\", \\"Kahramanmaraş\\")):\\n            iou = ort(d, il, \\"iou_hasar\\")\\n            referans_mi = egitim_bolgesi == ad\\n            s15.append({\\"Eğitim bölgesi\\": egitim_bolgesi, \\"Test bölgesi\\": ad, \\"Tohum\\": len(d), \\"IoU_hasar\\": virgul(iou),\\n                        \\"F₁\\": virgul(ort(d, il, \\"f1\\")), \\"Kesinlik\\": virgul(ort(d, il, \\"kesinlik\\")),\\n                        \\"Duyarlılık\\": virgul(ort(d, il, \\"duyarlilik\\")),\\n                        \\"Bölge içine göre fark\\": \\"— (referans)\\" if referans_mi else\\n                        (virgul(iou - ref[il]) if iou is not None and ref[il] is not None else \\"—\\"),\\n                        \\"Oran (bölge içine göre)\\": \\"—\\" if referans_mi or iou is None or not ref[il] else virgul(iou / ref[il], 2)})\\n    print(f\\"\\\\nTablo 15 — bölge içi ve bölgeler arası ({TABLO15_MODEL}, sınama)\\")\\n    print(yaz(pd.DataFrame(s15)))\\n    pd.DataFrame(s15).to_csv(f\\"{KOK_CIKTI}/{ESAS}_tablo15.csv\\", index=False)"}')
_kuyruk = list(KUYRUK)
for _n, _a in enumerate(_kuyruk, 1):
    _gecen = (time.time() - OTURUM_BASI) / 3600
    if _gecen + KOSU_PAYI_SA > 11.75:
        print(f"süre yetmez ({_gecen:.1f} sa geçti); sonraki oturuma kalan deneyler:", _kuyruk[_n - 1:], flush=True)
        break
    print("=" * 25, f"KUYRUK {_n}/{len(_kuyruk)}:", _a, f"| oturumda geçen {_gecen:.1f} sa", "=" * 25, flush=True)
    try:
        exec(_KAYNAK["ayar"])                        # varsayılanlar + oturum ayarları
        if DUMAN_TESTI:
            exec(os.environ.get("QM_AYAR", ""))      # yerel duman testinde ayar ezme
        globals().update(_a)
        for _k in ("0b", "4", "5", "6", "8"):
            exec(_KAYNAK[_k])
    except Exception:
        traceback.print_exc()
        print("HATA — bu deney atlandı, kuyruk sürüyor:", _a, flush=True)
print("KUYRUK BİTTİ")